# 📗 파인튜닝 데이터 만들기: 노동법 조문으로 노무 상담 Q&A 생성하기

노무 상담 챗봇을 파인튜닝하려면 모델이 따라 할 질문과 답의 예시가 필요합니다. 이 교안은 근로기준법 등 노동법 408개 문단 가운데 상담 질문이 나오는 382문단에서 LangChain 체인으로 Q&A를 만들고, 답의 숫자를 조문과 대조해 검수합니다. 행이 모자란 주제는 질문을 늘리고 검색으로 찾은 조문으로 답을 붙이며, 노무와 관계없는 범위 밖 질문에는 거절 답을 붙입니다. 마지막으로 모든 행을 JSONL(한 줄에 JSON 하나를 적는 파일) 파일로 저장합니다. 교안 02가 이 파일을 학습에 넣을 모양으로 다듬습니다.

<img src="images/00_lesson01_goals_1.png" width="1050" alt="제목은 '이번 교안의 핵심 1: 조문을 정리하고 Q&A를 만들기'입니다. 번호 배지 1~5의 카드 다섯 장이 화살표로 이어지고, 카드마다 아래에 '예' 칸이 있습니다. 1 '원문 정리'는 '개정 태그를 지우고 글자를 맞춤'이고, 예는 빨간 취소선을 그은 '<개정 2012.2.1>'이 화살표를 따라 '삭제'로 이어지는 그림입니다. 2 '생성 대상 고르기'는 '위임 문장과 목적 조항은 뺌'이고, 예는 빨강 엑스가 붙은 문서 카드들과 '408문단 중 382문단'입니다. 3 'Q&A 생성 체인'은 '조문 하나로 Q&A 1~3쌍, 구조화 출력'이고, 예는 말풍선 'Q 1년 80% 출근하면 연차는 며칠?'과 'A 15일'입니다. 4 '5개 문단으로 시험'은 '결과 확인과 비용 어림'이고, 예는 문서 카드 다섯 장과 '문단당 평균 비용 × 382'입니다. 5 '전체 생성'은 '.batch로 382문단을 동시에'이고, 예는 주황 카드 더미 옆의 빨강 카드 두 장과 '실패한 요청은 따로 모음'입니다. 맨 아래 띠에는 '조문 하나로 먼저 만들어 보고, 5개로 시험한 뒤 전체로 늘립니다'가 적혀 있습니다.">

<img src="images/00_lesson01_goals_2.png" width="1050" alt="제목은 '이번 교안의 핵심 2: 검수하고, 질문을 늘리고, 저장하기'입니다. 번호 배지 6~10의 카드 다섯 장이 화살표로 이어지고, 카드마다 아래에 '예' 칸이 있습니다. 6 '근거 검수'는 '숫자 대조와 금지 표현 검사'이고, 예는 '15일'에 초록 '통과', 빨간 동그라미를 친 '2일'에 빨강 '걸림'입니다. 7 '질문 늘리기'는 '시드 질문 3개로 상황이 다른 새 질문'이고, 예는 카드 몇 장이 주황 카드 더미로 늘어나는 그림과 '주제마다 10~30개', '자카드 0.8 이상은 버림'입니다. 8 '새 질문에 답 만들기'는 '검색한 조문 5개를 보고 답'이고, 예는 초록 카드 '답 + 근거 doc_id'와 빨강 엑스가 붙은 회색 카드 '없음이면 뺌'입니다. 9 '범위 밖 질문'은 '노무와 관계없는 질문 70개'이고, 예는 말풍선 '연말정산은 어떻게 하나요?'가 화살표를 따라 '거절 답' 말풍선으로 이어지는 그림입니다. 10 '비용 확인과 저장'은 '단계별 실제 비용, 통과한 행 저장'이고, 예는 막대그래프와 보라 파일 'labor_qa.jsonl'입니다. 맨 아래 띠에는 '숫자가 맞는 행만 남기고, 모자란 주제는 질문을 늘려 채운 뒤 파일로 남깁니다'가 적혀 있습니다.">

**학습 목표**

1. **원문 정리:** 개정 태그를 지우고 글자 모양을 맞춰, 조문을 생성용 LLM의 입력이자 검수 기준으로 씁니다.
2. **생성 대상 고르기:** 위임 문장 하나뿐인 문단과 목적 조항을 빼고 408문단 가운데 382문단을 고릅니다.
3. **Q&A 생성 체인:** 조문 하나로 Q&A 1~3쌍을 구조화 출력으로 받는 LangChain 체인을 만듭니다.
4. **5개 문단으로 시험:** 주제가 다른 5개 문단으로 결과를 확인하고 전체 비용을 어림합니다.
5. **전체 생성:** `.batch`로 382문단의 요청을 동시에 보내고, 실패한 요청은 따로 모읍니다.
6. **근거 검수:** 답의 숫자를 근거 조문과 대조하고(15일은 통과, 조문에 없는 2일은 걸림) 금지 표현을 검사합니다.
7. **질문 늘리기:** 시드 질문 3개로 상황이 다른 새 질문을 만들고, 자카드 0.8 이상으로 거의 같은 질문은 버립니다.
8. **새 질문에 답 만들기:** 하이브리드 검색으로 질문에 맞는 조문을 찾고, 생성용 LLM이 그 조문으로 답과 근거 doc_id를 씁니다.
9. **범위 밖 질문:** 노무와 관계없는 질문 70개에 정해 둔 거절 답을 붙입니다.
10. **비용 확인과 저장:** 단계별 실제 비용을 계산하고, 검수를 통과한 행을 `labor_qa.jsonl`로 저장합니다.

## 1. 파인튜닝 데이터

어떤 예시를 모을지 정하려면 파인튜닝으로 무엇이 바뀌는지부터 알아야 합니다.

**파인튜닝(fine-tuning)**
- 이미 학습된 모델을 우리 예시 데이터로 이어서 학습해 모델의 파라미터를 바꾸는 일입니다.
- 목적: 프롬프트에 매번 지시를 넣지 않아도 모델이 원하는 응답 방식(말투, 답의 길이, 모를 때의 답)을 늘 따르게 합니다.

같은 목적을 프롬프트나 RAG(retrieval-augmented generation: 검색한 문서를 근거로 넣어 답하게 하는 방법)로 이룰 수도 있습니다.

<img src="images/01_finetune_vs_prompt_rag.png" width="1050" alt="제목은 '같은 질문, 세 가지 방법: 프롬프트 · RAG · 파인튜닝'입니다. 서류철을 든 노무 담당자가 맨 위 말풍선 '1년 다녔는데 연차휴가는 며칠인가요?'를 가리킵니다. 아래에 판 세 개가 나란히 있습니다. '프롬프트' 판에서는 주황 지시 카드 '노동법에 근거해 존댓말로 짧게 답하라'가 질문 말풍선 앞에 붙고, 아래 띠에 '바뀌는 곳: 그 요청의 입력'이 있습니다. 'RAG' 판에서는 돋보기가 파랑 조문 카드 '근로기준법 제60조'를 찾아 질문 옆에 붙이고, 아래 띠에 '바뀌는 곳: 모델이 참고하는 문서'가 있습니다. '파인튜닝' 판에서는 초록 'Q&A 예시' 카드 더미가 모델 상자로 들어가 톱니바퀴가 바뀌고, 아래 띠에 '바뀌는 곳: 모델의 파라미터'가 있습니다.">

**알맞은 때**
- **프롬프트:** 말투나 형식을 가볍게 바꿀 때입니다. 지시가 길어질수록 요청마다 입력이 길어집니다.
- **RAG:** 법령 값처럼 바뀔 수 있는 사실을 답할 때입니다. 최신 조문을 검색해 넣으면 됩니다.
- **파인튜닝:** 응답 방식(존댓말로 짧게, 모르는 질문은 거절)을 지시 없이도 늘 같게 지킬 때입니다.
- 실무에서는 응답 방식은 파인튜닝으로, 바뀌는 최신 조문은 RAG로 함께 씁니다.

**Q&A(질문-답변 쌍)**
- 질문 하나와 그 답 하나를 묶은 학습 데이터 한 건입니다. 모델은 질문을 입력으로, 답을 따라 할 출력으로 배웁니다.

**좋은 파인튜닝 데이터의 조건**
- **양보다 품질:** [LIMA(Zhou et al., 2023)](https://arxiv.org/abs/2305.11206)는 잘 고른 예시 1,000개만으로 파인튜닝해도 모델이 답의 형식을 잘 배운다고 보고했습니다. [AlpaGasus(Chen et al., 2023)](https://arxiv.org/abs/2307.08701)는 LLM으로 만든 예시 5만 2천 개 가운데 품질이 높은 9천 개만 골라 학습한 모델이 전체로 학습한 모델보다 낫다고 보고했습니다.
- **응답 방식의 일관성:** LIMA는 모델의 지식은 대부분 사전학습에서 생기고, 적은 예시로 하는 파인튜닝은 주로 답하는 방식을 맞춘다는 가설을 세웠습니다. 파인튜닝으로 새 사실도 외우지만 느리게 학습되고, 새 사실을 많이 넣을수록 지어낸 답이 늘었다는 연구([Gekhman et al., 2024](https://arxiv.org/abs/2405.05904))가 있어, 모델이 모르는 사실이나 바뀌는 사실은 RAG로 넣습니다. 그래서 말투, 답의 길이, 근거를 지키는 방식, 모를 때의 답을 먼저 정하고 모든 행을 그 모양으로 맞춥니다.
- **실제 질문과 닮은 다양성:** 주제, 질문 유형, 말투가 실제 사용자 질문처럼 고르게 섞여야 합니다. 한쪽으로 몰린 데이터로 학습하면 적은 쪽 질문에 약해집니다.
- **검수:** LLM으로 만든 데이터에는 틀린 답이 섞이므로 답의 숫자가 근거 조문에 있는지 코드로 확인하는 자동 검사와 사람이 표본을 읽는 검토를 함께 합니다.
- **모르는 질문의 예시:** 답할 수 없는 질문에 거절하는 예시를 조금 넣어야 학습한 모델이 지어내 답하지 않습니다.
- **작게 시작:** [OpenAI 파인튜닝 가이드](https://developers.openai.com/api/docs/guides/supervised-fine-tuning)는 예시가 최소 10개 필요하고, 잘 만든 예시 50개로 시작해 결과를 보며 늘리라고 권합니다.

오늘 만드는 행은 학습할 때 질문과 답만 쓰고, 조문은 모델의 입력에 넣지 않습니다. 그래서 학습한 모델은 응답 방식과 함께 답 속의 법령 값도 외우게 되고, 앞의 연구처럼 지어낸 답이 늘 수 있습니다. 법령이 바뀌면 데이터를 다시 만들어 다시 학습해야 하고, 학습한 모델의 답도 조문과 다시 대조해 확인해야 합니다.

**오늘 만들 학습 데이터 한 행의 예**

| 항목 | 값 |
|---|---|
| 질문(`question`) | 1년 동안 80퍼센트 이상 출근했는데 연차휴가는 며칠인가요? |
| 답(`answer`) | 1년간 80퍼센트 이상 출근하셨다면 15일의 유급휴가를 받으실 수 있습니다. |
| 근거 조문(`doc_id`) | 근로기준법 제60조 제1항 |
| 주제 태그(`source`) | 근로시간·휴가 |
| 만든 방법(`method`) | 문서 생성(조문(문서)을 읽고 생성용 LLM이 바로 만든 Q&A) |
| 질문 유형(`question_type`) | 사실 확인 |

질문과 답은 생성용 LLM이 조문을 읽고 만들고, 나머지 칸은 어느 조문에서 어떤 방법으로 만든 행인지 기록합니다. 조문마다 이런 행을 1~3개씩 만들고, 답의 숫자가 조문과 맞는 행만 남깁니다.

## 2. 실습 준비

### 2-1. 생성용 LLM

학습 데이터를 만드는 데 쓰는 LLM을 생성용 LLM이라 부릅니다. 아래 셀은 경로와 API 키를 읽고 생성용 LLM `llm`을 만듭니다.

이 교안은 OpenAI API의 `gpt-5.6-luna`를 생성용 LLM으로 씁니다. GPU 파드의 로컬 LLM(Gemma 4 E4B, Ollama)으로 같은 데이터셋을 만드는 코드와 두 방법의 비용, 시간, 품질 비교는 `부록_로컬LLM_Gemma4로_데이터셋_만들기.ipynb`에 있습니다.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"
output_dir = material_dir / "output"  # 이 단원이 만든 데이터셋 파일을 저장하는 폴더
output_dir.mkdir(exist_ok=True)

# API 키 읽기: 단원 폴더 .env를 먼저 읽고, 없는 키만 상위 실습자료 .env에서 채웁니다
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")

# 데이터를 만드는 생성용 LLM. .env에 OPENAI_CHAT_MODEL이 없으면 기본 모델을 씁니다
# use_responses_api=True: 앞 단원들과 같이 OpenAI Responses API로 호출합니다
model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")
llm = ChatOpenAI(model=model_name, use_responses_api=True)

# 테이블의 긴 글(답, 조문)이 50자에서 잘리지 않고 끝까지 보이게 합니다
pd.set_option("display.max_colwidth", None)

print("생성용 LLM:", model_name)

### 2-2. 조문 읽기

Q&A의 근거가 될 법령 원문을 읽습니다.

- **데이터:** 근로기준법, 최저임금법, 근로자퇴직급여 보장법, 남녀고용평등과 일ㆍ가정 양립 지원에 관한 법률, 기간제 및 단시간근로자 보호 등에 관한 법률의 408문단입니다. 국가법령정보센터([law.go.kr](https://www.law.go.kr)) 법령 원문이고, 저작권법 제7조에 따라 보호받지 않는 저작물입니다. 법령별 시행일자는 `data/law_sources.json`에 있습니다.
- **키:** `doc_id`(법령·조·항, 예: "근로기준법 제60조 제1항"), `law`(법령 약칭), `source`(주제 태그), `text`(원문)입니다.
- **문단:** 조 하나 또는 항 하나입니다. 이 교안의 '조문 하나'는 문단 하나입니다.

In [ ]:
# 노동법 조문 문단: 문단마다 doc_id, law, source(주제 태그), text(원문)
law_articles = json.loads((data_dir / "본실습" / "labor_law_articles.json").read_text(encoding="utf-8"))
law_table = pd.DataFrame(law_articles)
print("문단 수:", len(law_articles))
display(law_table.head())

In [ ]:
# 주제 태그별 문단 수
source_counts = law_table["source"].value_counts()
display(source_counts)

**결과 읽기**
- **주제별 문단 수의 차이:** 모성보호·육아(78문단)와 취업규칙(7문단)은 열 배 넘게 차이 납니다. 이 차이가 주제별 행 수로 이어지므로, 뒤에서 행이 적은 주제의 질문을 더 만듭니다.

### 2-3. 원문 정리

조문은 생성용 LLM의 입력이고, 생성한 답을 검사하는 기준도 됩니다. 뒤에서 답에 나온 숫자가 근거 조문에도 있는지 비교해, 숫자를 틀리게 쓴 답을 거르기 때문입니다.

그래서 본문이 아닌 표시는 지우고, 다른 자료와 다르게 적힌 글자는 한 가지로 맞춥니다. 연차 유급휴가 조문으로 정리 전 원문과 원문의 숫자 리스트를 먼저 봅니다.

In [ ]:
import re

# doc_id로 조문을 바로 고르도록 딕셔너리(dict)를 만듭니다
article_by_id = {article["doc_id"]: article for article in law_articles}

# 연차 유급휴가 조문. 정리 전 원문은 숫자 대조에서 다시 쓰므로 따로 둡니다
para = article_by_id["근로기준법 제60조 제1항"]
para_raw_text = para["text"]
print(para_raw_text)

# 원문에서 숫자를 모두 뽑습니다
print("숫자:", re.findall(r"\d+", para_raw_text))

**결과 읽기**
- **개정 태그:** `<개정 2012.2.1>`처럼 조문이 바뀐 날짜를 꺾쇠로 적은 표시이고, 그 날짜 `'2012', '2', '1'`이 본문 숫자와 한 리스트에 섞입니다. 2가 섞여 있으면 15일을 "2일"로 틀리게 쓴 답도 숫자 대조를 통과하므로 태그를 지웁니다.
- **NFC 정규화:** 이 데이터의 '金'은 모양만 같은 호환용 한자(U+F90A)라 답에 쓴 일반 한자 '金'(U+91D1)과 다른 글자로 비교됩니다. 아래 `clean_text`가 `unicodedata.normalize("NFC", text)`로 표준 번호 하나로 바꿉니다.

In [ ]:
import unicodedata


def clean_text(text):
    """조문을 생성용 LLM의 입력이자 숫자 대조의 기준으로 쓸 수 있게 정리합니다.

    본문이 아닌 개정 태그는 지우고, 다르게 적힌 글자와 공백은 한 가지로 맞춥니다.
    """
    # NFC 정규화: 모양은 같지만 유니코드 번호가 다른 글자를 표준 번호 하나로 바꿉니다
    # 예: 조문의 '金'은 호환용 한자(U+F90A)라서, 답에 쓴 일반 한자 '金'(U+91D1)과 문자열 비교에서 다른 글자가 됩니다
    text = unicodedata.normalize("NFC", text)

    # 가운뎃점 자리에 쓴 한글 자모(U+318D)를 가운뎃점(U+00B7)으로 바꿉니다
    # 안 바꾸면 FAQ와 생성용 LLM의 답에 쓰인 가운뎃점과 다른 글자로 셉니다
    text = text.replace("\u318d", "\u00b7")

    # <개정 2012.2.1>, <신설 2008.3.21> 같은 개정 태그를 지웁니다. 시행일자는 교안 02의 데이터 카드에 적습니다
    # 안 지우면 날짜 2012, 2, 1 때문에 '2일' 같은 틀린 답이 숫자 대조를 통과합니다
    text = re.sub(r"<[^>]*>", "", text)

    # 줄바꿈과 겹친 공백을 공백 한 칸으로 맞춥니다(split()은 둘 다 구분자로 봅니다)
    # 안 맞추면 공백 모양만 다른 같은 문장을 다른 문장으로 봅니다
    return " ".join(text.split())

In [ ]:
# 같은 조문을 정리한 본문과 숫자 리스트
print(clean_text(para_raw_text))
print("숫자:", re.findall(r"\d+", clean_text(para_raw_text)))

In [ ]:
# 전체 문단을 정리하고, 정리 전에 개정 태그가 있던 문단 수와 정리로 본문이 바뀐 문단 수를 셉니다
changed_count = 0
tag_count = 0
for article in law_articles:
    # 정리 전 원문에 꺾쇠(<)로 시작하는 개정 태그가 있으면 셉니다
    if "<" in article["text"]:
        tag_count += 1

    # 정리한 본문이 원문과 다르면 셉니다
    cleaned = clean_text(article["text"])
    if cleaned != article["text"]:
        changed_count += 1
    article["text"] = cleaned
print("태그가 있던 문단 수:", tag_count, "/", len(law_articles))
print("정리로 바뀐 문단 수:", changed_count, "/", len(law_articles))

**결과 읽기**
- **181과 237:** 408문단 가운데 181문단(열에 넷꼴)에 개정 태그가 있어 숫자 리스트에 날짜가 섞여 있었고, 가운뎃점과 공백 정리까지 더해 237문단이 바뀌었습니다.
- **`para`:** `law_articles` 안의 같은 딕셔너리라 `para["text"]`도 정리한 본문이 됐고, 정리 전 원문은 `para_raw_text`에 남겨 뒤의 비교에 씁니다.

### 2-4. 생성 대상 문단 고르기

모든 문단에서 상담 질문이 나오지는 않습니다. 법의 목적만 적은 조항이나 세부 사항을 대통령령·부령에 넘기기만 하는 문단으로 Q&A를 만들면 "대통령령으로 정합니다" 같은 쓸모없는 답이 됩니다. `is_qa_target`이 이런 문단을 걸러 내고, 함께 따라하기의 주택임대차보호법 문단에도 같은 함수를 씁니다.

**걸러 내는 문단과 남기는 문단** (이 데이터의 실제 조문)

| 문단 | 본문 | 판정 |
|---|---|---|
| 근로기준법 제11조 제3항 | "…상시 사용하는 근로자 수를 산정하는 방법은 대통령령으로 정한다." | 제외: 위임 끝맺음으로 끝나는 한 문장뿐이라 답할 내용이 없음 |
| 근로기준법 제1조 | "제1조(목적) 이 법은 … 목적으로 한다." | 제외: 목적 조항 |
| 근로기준법 제74조 제1항 | "…90일의 출산전후휴가를 주어야 한다. 이 경우 … 고용노동부령으로 정한다." | 남김: 위임 문장 앞에 출산전후휴가 규칙 문장이 있음 |

- **한 문장인지 보는 방법:** 조문의 문장은 "~한다."처럼 "다."로 끝나므로, 문장이 둘 이상이면 앞 문장 끝의 "다. "(다, 마침표, 공백)가 본문 중간에 생깁니다. 그래서 "다. "가 없으면 문장이 하나뿐인 문단으로 봅니다. 호 목록("1. … 것.")처럼 "다."로 끝나지 않는 문장이 섞이면 틀릴 수 있는 간단한 규칙이지만, 이 데이터에서는 위임 끝맺음 문단을 모두 맞게 나눕니다.

`law_articles` 전체(408문단)는 답의 숫자를 근거 조문과 대조할 때와 새 질문에 답할 조문을 검색할 때 그대로 쓰고, Q&A 생성 입력에만 `qa_articles`를 씁니다.

In [ ]:
# 세부 내용을 시행령(대통령령)이나 시행규칙(고용노동부령) 등에 넘기는 문장의 끝맺음
DELEGATION_ENDINGS = ("대통령령으로 정한다.", "고용노동부령으로 정한다.", "정하는 바에 따른다.")


def is_qa_target(text):
    """Q&A를 만들 문단이면 True, 위임만 하는 한 문장이거나 목적 조항이면 False를 돌려줍니다."""
    # 1) 한 문장(중간에 "다. "가 없음)이고 위임 끝맺음으로 끝나면 제외합니다
    #    endswith에 튜플을 주면 튜플 안의 끝맺음 가운데 하나로 끝나는지 봅니다
    if "다. " not in text and text.endswith(DELEGATION_ENDINGS):
        return False

    # 2) 조 제목이 (목적)인 목적 조항은 제외합니다
    #    (의\d+)?는 "제43조의2"처럼 가지 번호가 붙은 조도 맞추려는 부분입니다
    if re.match(r"^제\d+조(의\d+)?\(목적\)", text):
        return False
    return True

In [ ]:
# 문단을 생성 대상과 제외한 문단으로 나눕니다
qa_articles = []
skipped_articles = []
for article in law_articles:
    if is_qa_target(article["text"]):
        qa_articles.append(article)
    else:
        skipped_articles.append(article)

print("생성 대상 문단 수:", len(qa_articles), "| 제외한 문단 수:", len(skipped_articles))
display(pd.DataFrame(skipped_articles))

**결과 읽기**
- **제외한 26문단:** 위임 끝맺음으로 끝나는 한 문장뿐인 문단 22개와 목적 조항 4개입니다.

## 3. 조문 하나로 데이터 한 행 만들기

전체 조문으로 늘리기 전에, 조문 하나에서 학습 데이터 한 행이 나오기까지를 끝까지 한 번 만듭니다.

### 3-1. 학습 데이터 포맷

LLM에 무엇을 만들게 할지 정하려면 학습 데이터 한 행의 모양(포맷)부터 정해야 합니다. 많이 쓰는 포맷은 두 가지입니다.

- **instruction 포맷:** 한 행을 `instruction`(지시: 모델에게 하는 요청 문장), `input`(추가 맥락, 없으면 빈 문자열), `output`(답) 세 칸에 담습니다. 한 번 묻고 한 번 답하는 데이터에 맞습니다.
- **chat 포맷:** 한 행을 역할이 붙은 메시지 리스트 `messages`에 담습니다. 역할은 `system`(챗봇의 역할), `user`(사용자 질문), `assistant`(모델의 답)이고, 여러 차례 이어지는 대화도 담을 수 있습니다.

<img src="images/02_two_formats.png" width="1050" alt="제목은 '학습 데이터 한 행의 두 포맷: instruction 포맷과 chat 포맷'입니다. 왼쪽 'instruction 포맷' 판에는 칸 세 개 'instruction: 1년 동안 80퍼센트 이상 출근했는데 연차휴가는 며칠인가요?', 'input: (빈 칸)', 'output: 15일의 유급휴가를 받으실 수 있습니다.'가 위에서 아래로 있습니다. 오른쪽 'chat 포맷(messages)' 판에는 말풍선 세 개 'system: 너는 노무 상담 챗봇이다.', 'user'(같은 질문), 'assistant'(같은 답)가 있습니다. 아래 띠에는 'chat 포맷은 system 문장(챗봇의 역할)을 담습니다'가 적혀 있습니다.">

**이 교안의 선택**
- 최종 학습 파일은 교안 02가 chat 포맷(`messages`)으로 만듭니다. 챗봇의 역할을 정하는 system 문장을 담을 수 있고, 학습한 모델이 실제 상담에서 받는 메시지 모양과 같기 때문입니다. 이 교안은 그 전 단계로 `question`·`answer` 키로 모읍니다.
- 다음 시간의 학습 도구 TRL(Hugging Face의 파인튜닝 라이브러리)은 아래 세 모양 가운데 하나로 된 데이터를 받습니다([TRL 데이터셋 포맷](https://huggingface.co/docs/trl/dataset_formats)). 같은 연차 Q&A 한 행을 세 모양으로 적으면 다음과 같습니다.

```text
# text: 질문과 답을 이어 붙인 학습 텍스트 하나
{"text": "질문: 1년 동안 80퍼센트 이상 출근했는데 연차휴가는 며칠인가요?\n답: 1년간 80퍼센트 이상 출근하셨다면 15일의 유급휴가를 받으실 수 있습니다."}

# messages: 역할이 붙은 메시지 리스트(이 교안이 쓰는 chat 포맷)
{"messages": [{"role": "system", "content": "너는 노무 상담 챗봇이다."},
              {"role": "user", "content": "1년 동안 80퍼센트 이상 출근했는데 연차휴가는 며칠인가요?"},
              {"role": "assistant", "content": "1년간 80퍼센트 이상 출근하셨다면 15일의 유급휴가를 받으실 수 있습니다."}]}

# prompt-completion: 모델에 주는 부분과 모델이 써야 할 답을 두 칸으로 나눔
{"prompt": "1년 동안 80퍼센트 이상 출근했는데 연차휴가는 며칠인가요?", "completion": "1년간 80퍼센트 이상 출근하셨다면 15일의 유급휴가를 받으실 수 있습니다."}
```

- instruction 포맷은 TRL이 바로 받는 모양이 아니라서 바꿔 넣습니다. `instruction`과 `input`을 이어 `prompt`(또는 user 메시지)로, `output`을 `completion`(또는 assistant 메시지)으로 옮깁니다.

### 3-2. Q&A 생성 체인

Q&A를 행으로 모으려면 생성용 LLM이 질문과 답을 정해진 칸에 담아 돌려줘야 합니다. 구조화 출력(Pydantic 모델로 정한 모양대로 답 받기, OpenAI API 활용 단원의 구조화 출력과 LangChain 에이전트·툴 단원의 `with_structured_output`)과 프롬프트 템플릿·`|` 체인(LangChain 기본 구조 단원)을 이어 씁니다.

- **질문 유형(`question_type`):** 직장인은 같은 조문을 사실 확인("연차는 며칠인가요"), 절차("연차는 언제까지 써야 하나요"), 문제 상황("회사가 연차를 안 줘요")으로 묻습니다. 한 유형만 있으면 모델이 다른 유형에 답하는 예시를 못 보므로 유형을 정해 섞게 합니다. 생성용 LLM이 붙인 라벨이라 틀릴 수 있어, 유형 분포를 대략 볼 때만 씁니다.

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field


class QAPair(BaseModel):
    """조문에 근거한 Q&A 한 건입니다."""
    question: str = Field(description="조문 내용만으로 답할 수 있는 직장인이나 사업주의 질문")
    answer: str = Field(description="조문에 적힌 내용으로 쓴 상담원의 한두 문장 답")
    # Literal[...]은 세 값 가운데 하나만 받는 타입입니다
    question_type: Literal["사실 확인", "절차", "문제 상황"] = Field(
        description="질문 유형. 사실 확인은 기준, 기간, 금액이 얼마인지 묻는 질문, 절차는 무엇을 언제 어떻게 해야 하는지 묻는 질문, 문제 상황은 회사가 이렇게 했는데 괜찮은지나 내가 받을 수 있는지 묻는 질문"
    )


class QASet(BaseModel):
    """조문 하나에서 만든 Q&A 리스트입니다."""
    items: list[QAPair] = Field(description="서로 다른 Q&A 리스트")

조문과 규칙을 담은 프롬프트를 만들고, 위의 `QASet` 모양으로만 답하는 체인으로 잇습니다.

- **system 메시지:** 생성용 LLM이 맡을 일과 그 결과가 어디에 쓰이는지(챗봇이 배울 예시)를 알려 줍니다. 결과의 쓰임을 알아야 학습 예시로 쓸 만한 질문과 답을 만듭니다.
- **human 메시지:** 무엇을 누구의 질문으로 만들지 먼저 적고, 규칙을 근거, 답 쓰는 법, 질문 쓰는 법, 조·항 번호로 묶은 뒤 맨 끝에 조문을 둡니다.
- **묻는 사람:** 노무 상담 챗봇에는 직장인뿐 아니라 사업주도 묻습니다. 평가 질문과 기존 FAQ에도 사업주의 질문이 섞여 있어, 질문을 직장인이나 사업주의 말로 쓰게 합니다.
- **개수 1~n:** 정확히 n개를 요청하면 물을 것이 적은 짧은 문단에서 억지 질문이 생기므로 1~n개 범위로 요청합니다. 스키마가 개수를 강제하지 않으므로 돌아오는 개수는 실행마다 달라집니다.
- **프롬프트 규칙이 막는 것:** 아래 프롬프트의 규칙 3은 조건(대상, 기간, 비율, 예외)을 뺀 틀린 답을 막습니다. 규칙 5는 같은 답이 학습 데이터에 반복되는 것을 막고, 규칙 6은 법률 문장을 그대로 따라 한 질문을 막습니다.
- **규칙 7(조·항 번호):** 사용자는 조문을 보지 않으므로 '제1항에 따라'처럼 번호만 적은 질문이나 답은 무엇을 가리키는지 알 수 없습니다. 그래서 질문에는 번호를 쓰지 않고, 답은 '근로기준법 제60조'처럼 법령 이름과 조 번호로만 조문을 밝힙니다. 조문 원문은 항 번호를 ⑦ 같은 기호로 적으므로, 답에 '제7항'을 쓰면 뒤의 숫자 대조에서 7 때문에 걸립니다.
- **숫자 대조를 통과하는 항·호 번호:** 본문에 '제1항제6호'처럼 숫자로 적힌 항·호 번호를 답에 옮기면, 같은 숫자가 조문에 있어 숫자 대조를 통과합니다. 이런 답도 사용자는 무엇을 가리키는지 알 수 없으므로, 생성 결과에서 답에 '제○항', '제○호'가 있는 행은 따로 읽어 확인합니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# system은 역할과 목적, human은 할 일, 목적별 규칙, 조문입니다({n}: 만들 개수의 상한, {paragraph}: 조문 본문)
# 사용자는 조문을 보지 않으므로 '조문에 따르면' 같은 말을 막습니다
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "노무 상담 챗봇이 배울 질문과 답을 써야 한다. 네가 쓴 질문과 답은 그대로 챗봇의 학습 예시가 된다."),
    ("human", """아래 [조문]으로, 직장인이나 사업주가 노무 상담 챗봇에 실제로 물어볼 Q&A를 1~{n}개 만들어라. 조문에서 물을 것이 적으면 적게 만든다.

근거:
1. [조문]에 적힌 내용만 쓰고, 조문에 없는 내용은 지어내지 않는다.

답 쓰는 법:
2. 답은 상담원이 직접 하는 존댓말로 한두 문장 쓰고, '조문에 따르면'·'조문상'·'이 조문'·'이 조항'·'해당 조항'·'해당 규정' 같은 말은 쓰지 않는다.
3. 답에는 조문에 적힌 조건(대상, 기간, 비율, 예외)을 빠뜨리지 않는다.

질문 쓰는 법:
4. 질문마다 유형(사실 확인, 절차, 문제 상황)을 정하고, 여러 개를 만들면 유형을 섞는다.
5. 같은 답이 나오는 질문은 하나만 만든다.
6. 질문은 법률 용어 대신 묻는 사람이 평소 쓰는 말로, 자기 상황을 말하며 묻는 문장으로 쓴다.

조·항 번호:
7. 질문에는 조·항 번호(제○조, 제○항)를 쓰지 않는다. 답에서 조문을 밝힐 때는 '근로기준법 제60조'처럼 법령 이름과 조 번호만 쓰고, 항·호 번호(제○항, 제○호)는 쓰지 않는다.

[조문]
{paragraph}"""),
])

# 프롬프트 -> 생성용 LLM(QASet 모양으로만 답함) 체인
# 입력 딕셔너리를 받아 QASet 객체를 돌려줍니다
qa_generator = qa_prompt | llm.with_structured_output(QASet)

**`qa_prompt.format_messages(...)`:** 템플릿의 빈자리를 채운 메시지 리스트를 돌려줍니다. 모델을 부르지 않으므로 비용 없이 실제로 보낼 내용을 확인할 수 있습니다.

In [ ]:
# 연차 조문으로 채운 메시지를 미리 봅니다(모델 호출 없음)
preview_messages = qa_prompt.format_messages(n=3, paragraph=para["text"])
for message in preview_messages:
    print(f"[{message.type}]")
    print(message.content)

### 3-3. 한 조문의 Q&A와 JSONL 한 줄

전체 조문에 보내기 전에 연차 조문 하나로 결과 모양을 확인합니다. 질문 유형 세 가지가 섞일 수 있도록 최대 3개(`n=3`)를 요청합니다.

In [ ]:
# 입력: 정리한 연차 조문 하나
print("입력 조문:", para["text"])

# 연차 조문 하나로 Q&A 1~3개를 요청합니다(모델 호출 1회)
qa_items = qa_generator.invoke({"n": 3, "paragraph": para["text"]}).items

# QAPair 리스트를 테이블로 보려고 딕셔너리 리스트로 바꿉니다
qa_item_table = pd.DataFrame([item.model_dump() for item in qa_items])
display(qa_item_table)

첫 Q&A에 근거 정보를 붙여 학습 데이터 한 행을 만듭니다.

- **`doc_id`:** 근거 조문입니다. 검수할 때 답과 대조할 조문을 이 값으로 찾습니다.
- **`source`:** 주제 태그입니다. 주제별 행 수를 볼 때 씁니다.
- **`method`:** 만든 방법입니다. "문서 생성"은 조문(문서)을 읽고 생성용 LLM이 바로 만든 Q&A이고, 뒤에서 질문 늘리기로 만든 새 질문에 생성용 LLM이 검색으로 찾은 조문을 보고 답을 쓴 행은 "질문 늘리기"로 구분합니다.
- **`question_type`:** 생성용 LLM이 붙인 질문 유형입니다. 유형별 행 수를 볼 때 씁니다.
- **JSONL(JSON Lines):** 한 줄에 JSON 객체 하나를 적는 파일 형식입니다. 한 줄이 한 행이라 행을 더하거나 한 줄씩 읽기 쉬워 학습 데이터 파일에 많이 씁니다.

In [ ]:
# 첫 Q&A에 근거 조문 정보를 붙여 학습 데이터 한 행을 만듭니다
first_row = {
    "id": "qa-0001",
    "doc_id": para["doc_id"],
    "source": para["source"],
    "method": "문서 생성",
    "question": qa_items[0].question,
    "answer": qa_items[0].answer,
    "question_type": qa_items[0].question_type,
}

# JSONL 한 줄: ensure_ascii=False 로 한글을 \uXXXX 대신 글자 그대로 씁니다
print(json.dumps(first_row, ensure_ascii=False))

## 4. 전체 조문으로 늘리기

생성 대상 382문단에 요청을 하나씩 차례로 보내면 응답을 기다리는 시간이 382번 쌓이므로, 요청을 한꺼번에 보냅니다.

### 4-1. 5개 문단으로 먼저 시험

전체를 보내기 전에 5개 문단으로 결과 모양, 걸린 시간, 토큰 수를 확인하고 생성 대상 전체 문단의 비용을 어림합니다. 주제가 다른 문단을 고르면 프롬프트가 여러 종류 조문에서 잘 도는지 한 번에 봅니다. 프롬프트가 잘못됐으면 5개 비용만 들이고 고칠 수 있습니다.

- **`.batch(입력 리스트, config={"max_concurrency": N})`:** 입력 리스트를 받아 요청을 동시에 보내고, 결과를 입력과 같은 순서의 리스트로 돌려줍니다(LangChain 기본 구조 단원).
- **`return_exceptions=True`:** 한 요청이 실패해도 나머지 결과를 받습니다. 실패한 자리에는 결과 대신 예외 객체가 들어가고, 기본값 `False`에서는 하나만 실패해도 `.batch` 전체가 에러로 멈춥니다.
- **`get_usage_metadata_callback()`:** `with` 블록 안에서 부른 채팅 모델의 토큰 사용량(`usage_metadata`)을 모델 이름별로 더해 줍니다. `input_tokens`(입력), `output_tokens`(출력), `total_tokens`(합)가 들어 있습니다.

In [ ]:
# 생성 대상 문단마다 체인 입력 딕셔너리를 만듭니다
qa_inputs = [{"n": 3, "paragraph": article["text"]} for article in qa_articles]

# 생성 대상 가운데 주제 태그가 다른 문단을 앞에서부터 하나씩 5개 모읍니다
trial_articles = []
seen_sources = []
for article in qa_articles:
    if article["source"] not in seen_sources:  # 처음 나온 주제 태그일 때만 고릅니다
        trial_articles.append(article)
        seen_sources.append(article["source"])
    if len(trial_articles) == 5:  # 주제가 다른 문단 5개가 모이면 멈춥니다
        break
trial_inputs = [{"n": 3, "paragraph": article["text"]} for article in trial_articles]

# 시험할 5개 문단(체인에 넣을 본문 text 포함)
display(pd.DataFrame(trial_articles))

In [ ]:
import time

from langchain_core.callbacks import get_usage_metadata_callback

# .batch가 동시에 보내는 요청 수의 상한입니다. 429 오류(요청 한도 초과)가 나면 줄입니다
MAX_CONCURRENCY = 32

# API를 부른 단계마다 토큰 사용량을 모으는 딕셔너리(키: 단계 이름). 마지막에 실제 비용을 계산할 때 씁니다
stage_usage = {}

# 5개 문단만 보내며 걸린 시간과 토큰 사용량을 기록합니다
start = time.perf_counter()
with get_usage_metadata_callback() as usage:
    trial_results = qa_generator.batch(trial_inputs, config={"max_concurrency": MAX_CONCURRENCY}, return_exceptions=True)
trial_seconds = time.perf_counter() - start  # 5개 요청이 모두 끝나기까지 걸린 초
stage_usage["5개 문단 시험"] = usage.usage_metadata

print("걸린 시간(초):", round(trial_seconds, 1))
display(usage.usage_metadata)

In [ ]:
# 5개 조문의 Q&A를 doc_id와 함께 테이블로 봅니다
# 실패한 요청(결과가 예외 객체)은 isinstance로 가려 건너뛰고 수만 셉니다
trial_rows = []
trial_failed_count = 0
for article, result in zip(trial_articles, trial_results):
    if isinstance(result, Exception):
        trial_failed_count += 1
    else:
        for item in result.items:
            trial_rows.append({"doc_id": article["doc_id"], "question_type": item.question_type, "question": item.question, "answer": item.answer})
print("실패한 요청 수:", trial_failed_count)
trial_table = pd.DataFrame(trial_rows)
display(trial_table)

In [ ]:
def sum_total_tokens(usage_metadata):
    """모델 이름별 usage_metadata의 total_tokens를 모두 더해 돌려줍니다."""
    total = 0
    for model_usage in usage_metadata.values():
        total += model_usage["total_tokens"]
    return total


# 토큰은 요청 수에 비례하므로 시험 문단 하나의 평균 토큰 수에 전체 문단 수를 곱해 어림합니다
# 실패한 요청의 토큰은 세지지 않아도 시험 문단 수로 나누므로, 실패가 있으면 이 어림은 실제보다 작게 나옵니다
estimated_tokens = round(sum_total_tokens(stage_usage["5개 문단 시험"]) / len(trial_inputs) * len(qa_inputs))  # 요청당 평균 토큰 수 x 전체 요청 수
print("전체 예상 토큰 수:", estimated_tokens)

# 요청은 동시에 최대 MAX_CONCURRENCY개가 처리되고, 하나가 끝나면 다음 요청이 들어갑니다. 5개 시험 시간에 (전체 문단 수 / MAX_CONCURRENCY)를 곱해 어림합니다
estimated_seconds = round(trial_seconds * len(qa_inputs) / MAX_CONCURRENCY)  # 32개씩 동시에 처리된다고 본 어림
print("전체 예상 시간(초):", estimated_seconds)

**결과 읽기**
- **테이블:** 질문과 답이 프롬프트 규칙(존댓말 한두 문장, '조문에 따르면' 없음, 조건 유지, 유형 섞임, 조·항 번호)을 지키는지 읽고, 어긋나면 프롬프트를 고쳐 다시 시험합니다.
- **예상 토큰 수와 시간:** 규모를 보는 어림값입니다. 생성용 LLM의 토크나이저로 센 값이라, 학습할 모델의 토큰 수는 교안 02에서 그 모델의 토크나이저로 다시 셉니다.
- **`usage_metadata`의 키:** 응답에 찍힌 모델 이름이라 `model_name`과 다를 수 있습니다.

**예상 비용**

OpenAI API는 입력 토큰과 출력 토큰에 가격을 따로 매깁니다. 시험 5문단에 든 비용을 `usage_cost`로 구해 문단 하나의 평균 비용을 내고, 생성 대상 문단 수를 곱해 조문으로 Q&A 생성 전체의 비용을 어림합니다.

- **가격(`gpt-5.6-luna`, 토큰 100만 개당):** 입력 0.20달러, 캐시 입력(앞 요청과 앞부분이 같아 캐시에서 읽은 입력) 0.02달러, 출력 1.20달러입니다. 가격은 바뀌므로 [OpenAI API 가격](https://developers.openai.com/api/docs/pricing) 페이지에서 확인하고 아래 상수를 고칩니다.
- **출력 토큰:** 답 글자뿐 아니라 모델이 답하기 전에 쓰는 추론 토큰도 들어 있고, 모두 출력 가격으로 셉니다.

In [ ]:
# gpt-5.6-luna 가격(달러, 토큰 100만 개당). 가격은 바뀌므로 OpenAI 가격 페이지에서 확인합니다
INPUT_PRICE = 0.20
CACHED_INPUT_PRICE = 0.02
OUTPUT_PRICE = 1.20


def usage_cost(usage_metadata):
    """모델 이름별 usage_metadata의 토큰 수에 가격을 곱한 비용(달러)을 돌려줍니다."""
    cost = 0
    for model_usage in usage_metadata.values():
        # 입력 토큰 가운데 캐시에서 읽은 토큰은 캐시 입력 가격, 나머지는 입력 가격으로 셉니다
        cached = model_usage.get("input_token_details", {}).get("cache_read", 0)
        cost += (model_usage["input_tokens"] - cached) * INPUT_PRICE
        cost += cached * CACHED_INPUT_PRICE

        # 출력 토큰(추론 토큰 포함)은 출력 가격으로 셉니다
        cost += model_usage["output_tokens"] * OUTPUT_PRICE
    return cost / 1000000


# 시험 5문단에 든 비용으로 문단 하나의 평균 비용을 구하고, 생성 대상 전체 문단의 비용을 어림합니다
cost_per_paragraph = usage_cost(stage_usage["5개 문단 시험"]) / len(trial_inputs)  # 문단 하나의 평균 비용(달러)
estimated_cost = cost_per_paragraph * len(qa_inputs)  # 문단당 평균 비용 x 생성 대상 문단 수
print("문단당 평균 비용(달러):", round(cost_per_paragraph, 5), "| 전체 문단 예상 비용(달러):", round(estimated_cost, 4))

**결과 읽기**
- **어림값:** 시험 5문단의 평균이라 실제와 다를 수 있고, 실패한 요청은 비용이 기록되지 않아 실패가 있으면 작게 어림합니다.
- **뒤 단계:** 질문 늘리기, 새 질문에 답 만들기, 범위 밖 질문 만들기는 요청 수가 앞 단계 결과로 정해져 미리 어림하지 않고, 마지막 API 비용 확인에서 실제 비용만 봅니다.

### 4-2. 생성 대상 382문단

5개 문단의 결과가 괜찮으면 같은 호출을 생성 대상 전체 입력으로 보냅니다. 요청이 382개면 일부가 시간 초과나 요청 한도 초과(429 오류)로 실패할 수 있습니다. 5개 문단으로 시험할 때와 같이 `return_exceptions=True`로 실패한 요청을 예외 객체로 받고, 이번에는 실패한 문단의 `doc_id`와 예외를 따로 모읍니다.

In [ ]:
# 전체 문단을 보냅니다. 실패한 요청은 결과 자리에 예외 객체로 돌려받습니다
start = time.perf_counter()
with get_usage_metadata_callback() as usage:
    qa_results = qa_generator.batch(qa_inputs, config={"max_concurrency": MAX_CONCURRENCY}, return_exceptions=True)
qa_seconds = time.perf_counter() - start
stage_usage["조문으로 Q&A 생성"] = usage.usage_metadata

# 5개 문단으로 어림한 값과 실제 값을 나란히 봅니다
print("토큰 수 예상:", estimated_tokens, "| 실제:", sum_total_tokens(usage.usage_metadata))
print("걸린 시간(초) 예상:", estimated_seconds, "| 실제:", round(qa_seconds, 1))
print("비용(달러) 예상:", round(estimated_cost, 4), "| 실제:", round(usage_cost(usage.usage_metadata), 4))  # 실제: 이 단계 사용량 x 가격

In [ ]:
# 성공한 결과는 행으로 펼치고, 실패한 요청은 doc_id와 예외를 따로 모읍니다
qa_rows = []
failed_rows = []
for article, result in zip(qa_articles, qa_results):
    if isinstance(result, Exception):
        failed_rows.append({"doc_id": article["doc_id"], "error": repr(result)})
    else:
        for item in result.items:
            qa_rows.append({
                # :04d 는 네 자리가 되도록 앞을 0으로 채웁니다(1 -> 0001)
                "id": f"qa-{len(qa_rows) + 1:04d}",
                "doc_id": article["doc_id"],
                "source": article["source"],
                "method": "문서 생성",
                "question": item.question,
                "answer": item.answer,
                "question_type": item.question_type,
            })

# 실패한 문단(열 이름을 정해 두면 0건이어도 빈 테이블이 만들어집니다)
failed_table = pd.DataFrame(failed_rows, columns=["doc_id", "error"])
print("실패한 요청 수:", len(failed_rows))
display(failed_table)

In [ ]:
# Q&A 행 테이블
qa_table = pd.DataFrame(qa_rows)
print("Q&A 행 수:", len(qa_rows))
display(qa_table.head())

# 주제 태그별 생성 대상 문단 수와 Q&A 행 수를 나란히 놓고, 한 주제가 모자라지 않은지 봅니다
qa_source_counts = pd.DataFrame(qa_articles)["source"].value_counts()  # 생성 대상 문단만 센 주제별 수
topic_coverage = pd.DataFrame({"문단 수": qa_source_counts, "Q&A 행 수": qa_table["source"].value_counts()})
display(topic_coverage)

In [ ]:
# 질문 유형별 행 수
display(qa_table["question_type"].value_counts())

# 문단당 Q&A 수: doc_id별 행 수를 다시 세면 "Q&A가 n개인 문단이 몇 개인지"가 됩니다
qa_per_article = qa_table["doc_id"].value_counts()
display(qa_per_article.value_counts().sort_index())

**결과 읽기**
- **예상과 실제:** 크게 어긋나면 5개 문단이 전체를 대표하지 못한 것이라 다음에는 시험 문단을 늘립니다. 시간만 크게 늘었으면 요청 한도(429)에 걸린 것이라 `MAX_CONCURRENCY`를 줄입니다.
- **실패한 요청:** 이 교안은 실패한 문단을 빼고 진행하고, 실패한 문단만 다시 보내는 방법은 함께 따라하기에서 구현합니다.
- **분포:** 문서 생성 행 수가 중앙값보다 적은 주제는 뒤에서 질문 늘리기로 채우고, 질문 유형이 한쪽으로 몰리면 프롬프트 규칙 4를 고칩니다. 문단당 Q&A 수가 모두 3개면 억지로 채웠는지, 대부분 1개면 조건을 빠뜨렸는지 읽어 봅니다.

## 5. 근거 검수

생성용 LLM은 조문에 없는 내용을 지어내거나 숫자를 바꿔 쓸 수 있습니다. 노무 상담 답의 핵심은 일수, 비율, 시간 같은 숫자라서, 답의 숫자가 근거 조문에 있는지부터 확인합니다.

### 5-1. 숫자 대조

- **근거 검수:** 생성된 답이 근거 조문에 있는 내용인지 확인하는 일입니다.
- **`numbers_match(answer, paragraph)`:** 답과 조문에서 `re.findall(r"\d+", ...)`로 숫자를 뽑고, 답의 숫자가 모두 조문의 숫자 리스트에 있으면 `True`를 돌려줍니다.

<img src="images/03_doc_to_qa_check.png" width="1050" alt="제목은 '조문에서 Q&A를 만들고 숫자로 검수합니다'입니다. 번호 배지 1, 2, 3의 세 단계가 화살표로 이어집니다. 1단계에는 파랑 조문 카드 '사용자는 1년간 80퍼센트 이상 출근한 근로자에게 15일의 유급휴가를 주어야 한다.'가 있습니다. 그 아래 회색 칩 '<개정 2012.2.1>'에는 빨간 취소선이 그어져 있고, 계산 결과 상자 '숫자: 60, 1, 80, 15'가 이어집니다. 2단계 'LLM 생성'에서는 답 카드 '15일의 유급휴가를 받으실 수 있습니다.'와 '2일의 유급휴가를 받으실 수 있습니다.'가 나옵니다. 3단계 '숫자 대조'에서는 15일 카드가 초록 '통과', 빨간 동그라미를 친 2일 카드가 빨강 '걸림'입니다.">

연차 조문(제60조, 1년간, 80퍼센트, 15일)에 네 종류의 답(조문을 옮긴 답, 표현이 다른 맞는 답, 15일을 10일로 바꾼 답, 15일을 개정 태그에만 있는 2일로 바꾼 답)을 정리 전 원문과 정리한 본문에 각각 대 봅니다.

In [ ]:
def numbers_match(answer, paragraph):
    """답에 나온 숫자가 모두 조문에도 있으면 True를 돌려줍니다."""
    # 조문의 숫자 리스트
    paragraph_numbers = re.findall(r"\d+", paragraph)

    # 답의 숫자 가운데 하나라도 조문에 없으면 False
    for number in re.findall(r"\d+", answer):
        if number not in paragraph_numbers:
            return False
    return True


# 연차 조문에 손으로 쓴 답 네 개
sample_answers = [
    {"답 종류": "조문을 옮긴 답", "답": "1년간 80퍼센트 이상 출근한 근로자에게는 15일의 유급휴가를 주어야 합니다."},
    {"답 종류": "표현이 다른 맞는 답", "답": "1년 동안 80퍼센트 이상 출근하셨다면 연차 유급휴가 15일을 받으실 수 있습니다."},
    {"답 종류": "15일을 10일로 바꾼 틀린 답", "답": "1년간 80퍼센트 이상 출근하셨다면 10일의 유급휴가를 받으실 수 있습니다."},
    {"답 종류": "15일을 2일로 바꾼 틀린 답", "답": "1년간 80퍼센트 이상 출근하셨다면 2일의 유급휴가를 받으실 수 있습니다."},
]
sample_table = pd.DataFrame(sample_answers)

# 같은 답을 정리 전 원문과 정리한 본문에 각각 대조합니다
sample_table["정리 전 원문과 대조"] = [numbers_match(answer, para_raw_text) for answer in sample_table["답"]]
sample_table["정리한 본문과 대조"] = [numbers_match(answer, para["text"]) for answer in sample_table["답"]]
display(sample_table)

**결과 읽기**
- **2일로 바꾼 답:** 정리 전 원문과 대조하면 개정 태그의 날짜 2 때문에 통과하고, 정리한 본문과 대조해야 걸립니다. 그래서 전체 검수는 정리한 본문과 대조합니다.

### 5-2. 전체 Q&A 검수

같은 대조를 `qa_rows` 전체에 적용합니다. 행마다 `doc_id`로 정리한 근거 조문을 찾아 답과 대조합니다.

In [ ]:
# doc_id로 정리한 본문을 바로 찾도록 딕셔너리를 만듭니다
doc_texts = {article["doc_id"]: article["text"] for article in law_articles}

# 행마다 근거 조문과 숫자를 대조한 결과를 열로 붙이고, 통과(True)와 걸림(False) 건수를 셉니다
qa_table["숫자 대조"] = [numbers_match(row["answer"], doc_texts[row["doc_id"]]) for row in qa_rows]
display(qa_table["숫자 대조"].value_counts())

In [ ]:
# 걸린 행만 골라 답과 근거 조문, 양쪽의 숫자 리스트를 나란히 놓고 사람이 비교합니다
mismatch_rows = []
for row in qa_rows:
    paragraph = doc_texts[row["doc_id"]]
    if not numbers_match(row["answer"], paragraph):
        mismatch_rows.append({
            "doc_id": row["doc_id"],
            "answer": row["answer"],
            "근거 조문": paragraph,
            "답의 숫자": re.findall(r"\d+", row["answer"]),
            "조문의 숫자": re.findall(r"\d+", paragraph),
        })
mismatch_table = pd.DataFrame(mismatch_rows, columns=["doc_id", "answer", "근거 조문", "답의 숫자", "조문의 숫자"])
display(mismatch_table.head(10))

**결과 읽기**
- 걸린 행은 저장할 때 모두 빠지므로, 위 테이블에서 근거 조문과 비교해 틀린 답이 얼마나 되는지 봅니다.
- 맞는 답도 걸릴 수 있습니다: 단위를 바꾼 답(1년 -> 12개월), 질문의 숫자를 되짚은 답, 조문 숫자로 계산한 답(35 + 5 = 40시간), 항 번호를 적은 답(원문은 ⑦처럼 기호로 적어 7이 없음).

**숫자 대조로 못 잡는 할루시네이션과 누락**
- **숫자 없는 사실을 바꾼 답:** "유급휴가"를 "무급휴가"로 바꾼 답은 숫자가 같아 통과합니다.
- **조문 안의 다른 숫자를 쓴 답:** 조 번호 60을 가져와 "60일의 유급휴가"라고 쓴 답도 60이 조문에 있어 통과합니다.
- **숫자를 글로 쓴 답, 조문에 없는 조언:** '열닷새'처럼 글로 쓴 숫자나 "인사팀에 서면으로 신청하세요" 같은 덧붙인 절차는 뽑을 숫자가 없어 통과합니다.
- **누락(할루시네이션은 아님):** 상시 4명 이하 사업장 제외처럼 다른 조문의 조건이 빠진 답은 조문 하나 단위로 만드는 방법의 한계라, 교안 02의 데이터 카드 한계에 적습니다.

이런 답은 사람이 표본을 읽어 확인합니다.

### 5-3. 금지 표현 검사

프롬프트에 적은 규칙은 지켜지지 않을 수 있으므로, 생성한 뒤 코드로 다시 확인합니다. 사용자는 조문을 보지 않으므로 '조문에 따르면' 같은 말이 답에 남으면 학습한 모델도 그 말투를 따라 합니다.

`BANNED_PHRASES`의 마지막 네 표현은 뒤의 새 질문에 답 만들기에서 생성용 LLM이 질문의 일부만 답하고 붙이는 거절 문장입니다. 근거 있는 답에 이런 문장이 섞이면 학습한 모델이 답할 수 있는 질문에도 일부를 거절하므로, 같은 검사로 거릅니다. 정해 둔 거절 답을 붙이는 범위 밖 행은 이 검사를 거치지 않고 저장할 때 따로 더합니다.

In [ ]:
# 프롬프트 규칙 2번에서 막은 표현과, 생성용 LLM이 일부만 답하고 붙이는 거절 문장('안내해 드릴 수 없', '제공된 조문', '정해져 있지 않', '판단할 수 없')
BANNED_PHRASES = ["조문에 따르면", "조문상", "이 조문", "이 조항", "해당 조항", "해당 규정", "안내해 드릴 수 없", "제공된 조문", "정해져 있지 않", "판단할 수 없"]


def has_banned_phrase(answer):
    """답에 막은 표현이 하나라도 있으면 True를 돌려줍니다."""
    for phrase in BANNED_PHRASES:
        if phrase in answer:
            return True
    return False


# 행마다 막은 표현이 있는지 열로 붙이고, 걸린 행을 봅니다
qa_table["금지 표현"] = [has_banned_phrase(row["answer"]) for row in qa_rows]
display(qa_table["금지 표현"].value_counts())
display(qa_table[qa_table["금지 표현"] == True].head())

**결과 읽기**
- 걸린 행은 저장할 때 숫자 대조에 걸린 행과 함께 빠집니다. 많으면 프롬프트 규칙 2를 고치고 5개 문단으로 다시 시험합니다.

## 6. 질문 늘리기

문서에서 만든 질문은 조문 하나에서 나오므로 내용과 표현이 그 조문의 범위를 벗어나기 어렵습니다. 앞에서 연차 조문으로 만든 질문 테이블(`qa_item_table`)에서 확인할 수 있습니다(실행마다 다름). 직장인이나 사업주는 조문 하나에 맞춰 묻지 않고 자기 상황을 말하며 물으므로, 그런 질문을 따로 늘립니다.

**질문 늘리기**
- 사람이 쓴 시드 지시(seed instruction: 처음에 손으로 쓴 예시 지시) 몇 개를 LLM에 보여 주고, 표현과 상황이 다른 새 지시를 만들게 하는 방법입니다. 지시(instruction)는 모델에게 하는 요청 문장이고, 이 교안에서는 직장인이나 사업주의 질문입니다.
- [Wang et al.(2022)](https://arxiv.org/abs/2212.10560)은 시드 지시 175개로 시작해 생성과 거르기를 반복하며 지시를 5만 2천여 개로 늘렸습니다.
- 원 논문은 같은 모델이 만든 데이터로 그 모델을 다시 학습시켰습니다. [Stanford Alpaca](https://crfm.stanford.edu/2023/03/13/alpaca.html)는 큰 모델로 지시와 답을 만들고, 그 데이터로 작은 모델(LLaMA 7B)을 학습시켰습니다. 이 교안도 생성용 LLM으로 질문과 답을 만드는 데까지는 같고, 이 데이터로 하는 파인튜닝은 다음 시간에 합니다.
- **새 질문과 답:** 질문 늘리기는 상황이 다른 새 질문만 만들고 답은 만들지 않습니다. 묻는 상황이 다르면 답도 달라지므로, 답은 뒤에서 생성용 LLM이 검색으로 찾은 조문을 보고 새로 씁니다.
- **교안 02의 패러프레이즈와 다른 점:** 패러프레이즈는 같은 질문을 다른 표현으로 바꾸고 원래 답을 그대로 붙이므로 뜻이 바뀌면 안 됩니다. 질문 늘리기의 새 질문은 뜻이 달라야 하므로, 거의 같은 질문은 자카드 0.8 기준으로 버립니다.

<img src="images/04_question_answer.png" width="1050" alt="제목은 '질문을 늘리고, 생성용 LLM이 조문으로 답을 만듭니다'입니다. 윗줄 '질문 늘리기'에서는 '시드 지시 3개'가 '생성용 LLM'을 거쳐 '주제마다 새 지시 10~30개' 카드 더미가 됩니다. 이 더미가 '자카드 0.8 이상이면 버림' 상자를 지나면 빨강 엑스 카드가 빠지고 '남은 지시' 더미가 남습니다. 아랫줄 '새 질문에 답 만들기'에서는 '남은 지시' 카드에서 나온 화살표가 연노랑 칩 '하이브리드 검색 + 재정렬'을 지나 파랑 조문 카드 '검색으로 찾은 조문 5개'로 이어지고, '검색으로 찾은 조문 5개'와 '남은 지시'가 '생성용 LLM'으로 들어갑니다. 그 결과 'doc_id' 배지가 붙은 초록 '답 + 근거 doc_id' 카드가 나오고, 빨강 엑스를 친 회색 'doc_id 없음' 카드는 아래로 빠집니다.">

### 6-1. 시드 지시로 새 지시 만들기

- **주제별 요청 개수:** 주제별 문서 생성 행 수의 중앙값을 목표로 두고, 목표에 모자란 만큼 새 지시를 요청합니다. 하한 `MIN_NEW_PER_TOPIC`(10)은 행이 많은 주제에도 자기 상황을 말하는 질문을 조금은 섞으려는 값이고, 상한 `MAX_NEW_PER_TOPIC`(30)은 문단이 적은 주제(취업규칙 7문단)에서 같은 조문 몇 개로 질문을 많이 만들면 비슷한 질문만 늘어나기 때문에 둡니다. 두 값은 이 교안이 정한 값이고, 교안 02에서 주제 분포를 확인한 뒤 조정합니다.

In [ ]:
class InstructionSet(BaseModel):
    """새로 만든 지시 리스트입니다."""
    items: list[str] = Field(description="서로 다른 새 지시 문장 리스트")


# 손으로 쓴 시드 지시 3개(연차휴가, 수습 해고, 배우자 출산전후휴가)
seed_instructions = [
    "입사한 지 1년이 안 됐는데 연차휴가가 생기나요?",
    "수습 기간이 끝나기 전에 회사가 그만 나오라고 하면 해고인가요?",
    "아내가 출산 예정인데 배우자 출산전후휴가는 며칠 쓸 수 있나요?",
]

# {topic}(주제 태그), {seeds}(시드 지시 줄), {n}(만들 개수)은 부를 때 채웁니다
question_prompt = ChatPromptTemplate.from_messages([
    ("system", "직장인이나 사업주가 노무 상담 챗봇에 실제로 입력할 법한 질문을 써야 한다. 네가 쓴 질문은 챗봇 학습 데이터의 질문이 된다."),
    ("human", """직장인이나 사업주가 노무 상담 챗봇에 물어볼 '{topic}' 주제의 새 지시 {n}개를 만들어라. 지시는 챗봇에 입력하는 질문이고, 아래 [시드 지시]는 그 예시다.

질문 쓰는 법:
1. 시드 지시와 표현, 상황을 다르게 쓰고, 시드 지시를 그대로 옮기거나 낱말만 바꾸지 않는다.
2. 사실 확인, 절차, 문제 상황 질문을 섞는다.
3. 묻는 사람이 자기 상황을 말하며 묻는 말투로 쓴다.

[시드 지시]
{seeds}"""),
])

# 프롬프트 -> 생성용 LLM(InstructionSet 모양으로만 답함) 체인
instruction_generator = question_prompt | llm.with_structured_output(InstructionSet)

In [ ]:
# 시드 지시를 "- 지시" 줄로 잇습니다(모든 주제에 같은 시드)
seed_lines = "\n".join([f"- {seed}" for seed in seed_instructions])
print("[시드 지시] 자리에 들어갈 줄:")
print(seed_lines)

# 주제별 요청 개수: 문서 생성 행 수가 중앙값(목표)에 모자란 만큼 요청하되, 10개 이상 30개 이하로
MIN_NEW_PER_TOPIC = 10
MAX_NEW_PER_TOPIC = 30

# 주제별 문서 생성 행 수와 그 중앙값
topic_row_counts = qa_table["source"].value_counts()
median_rows = topic_row_counts.median()  # 주제마다 채울 목표 행 수

# 주제 태그 12개마다 입력 딕셔너리를 만듭니다
topics = list(source_counts.index)
instruction_inputs = []
for topic in topics:
    # 1) 목표(중앙값)까지 모자란 행 수
    n = int(median_rows - topic_row_counts.get(topic, 0))  # 행이 없는 주제는 get이 0을 돌려줌

    # 2) 큰 주제도 자기 상황을 말하는 질문을 조금은 섞도록 하한, 문단이 적은 주제에 비슷한 질문이 몰리지 않도록 상한
    if n < MIN_NEW_PER_TOPIC:
        n = MIN_NEW_PER_TOPIC
    if n > MAX_NEW_PER_TOPIC:
        n = MAX_NEW_PER_TOPIC
    instruction_inputs.append({"topic": topic, "seeds": seed_lines, "n": n})

# 주제별 문서 생성 행 수와 요청 개수를 확인합니다(seeds 칸은 모두 같아 뺍니다)
instruction_plan = pd.DataFrame(instruction_inputs)[["topic", "n"]]
instruction_plan["문서 생성 행 수"] = [topic_row_counts.get(topic, 0) for topic in topics]
display(instruction_plan)

In [ ]:
# 주제별 입력을 한꺼번에 보냅니다(호출 12회). 실패한 요청은 결과 자리에 예외 객체로 돌려받습니다
# 토큰 사용량은 마지막에 실제 비용을 계산하도록 stage_usage에 남깁니다
with get_usage_metadata_callback() as usage:
    instruction_results = instruction_generator.batch(instruction_inputs, config={"max_concurrency": MAX_CONCURRENCY}, return_exceptions=True)
stage_usage["질문 늘리기"] = usage.usage_metadata

# 지시마다 요청한 주제 태그를 붙여 행으로 모읍니다. 실패한 요청은 건너뛰고 수만 셉니다
new_instructions = []
instruction_failed_count = 0
for topic, result in zip(topics, instruction_results):
    if isinstance(result, Exception):
        instruction_failed_count += 1
    else:
        for instruction in result.items:
            new_instructions.append({"source": topic, "question": instruction})

new_instruction_table = pd.DataFrame(new_instructions)
print("실패한 요청 수:", instruction_failed_count)
print("새 지시 수:", len(new_instructions))
display(new_instruction_table.head())

**결과 읽기**
- **요청 개수 테이블:** 주제마다 `n`이 10~30 사이이고, 앞에서 생성한 주제별 행 수에 따라 실행마다 달라집니다.
- **실패한 요청 수:** 0이 아니면 그 주제의 새 지시가 빠진 것이므로, 실패한 주제의 입력만 다시 보냅니다.

### 6-2. 비슷한 지시 거르기

새 지시가 시드나 앞서 남긴 지시를 거의 그대로 옮긴 문장이면 데이터에 더해도 다양성이 늘지 않으므로 버립니다.

**자카드 유사도(Jaccard similarity)**
- 두 집합의 교집합 크기를 합집합 크기로 나눈 값으로, 0(겹침 없음)~1(같은 집합)입니다.
- 이 교안은 두 문장에서 공백을 뺀 글자 집합 $A$, $B$를 비교합니다.

$\displaystyle J(A, B) = \frac{|A \cap B|}{|A \cup B|}$

첫 시드 지시와 손으로 쓴 지시 세 개로 값의 크기를 봅니다.

In [ ]:
def jaccard(a, b):
    """공백을 뺀 두 문장의 글자 집합으로 계산한 자카드 유사도를 돌려줍니다."""
    chars_a = set(a.replace(" ", ""))
    chars_b = set(b.replace(" ", ""))
    return len(chars_a & chars_b) / len(chars_a | chars_b)


# 첫 시드 지시와 끝맺음만 바꾼 지시, 같은 뜻의 다른 표현, 다른 주제의 지시를 비교합니다
jaccard_rows = []
for other in ["입사한 지 1년이 안 됐는데 연차휴가가 생겨요?", "아직 근속 1년 미만인데 유급휴가를 받을 수 있을까요?", "수습 기간에도 최저임금을 다 받아야 하나요?"]:
    jaccard_rows.append({"시드 지시": seed_instructions[0], "비교 지시": other, "자카드": round(jaccard(seed_instructions[0], other), 2)})
jaccard_table = pd.DataFrame(jaccard_rows)
display(jaccard_table)

**결과 읽기**
- **0.20:** 같은 뜻을 다른 낱말로 쓴 지시가 끝맺음만 바꾼 지시(0.86)보다 다른 주제의 지시(0.11)에 더 가깝습니다. 자카드는 글자 겹침만 보므로 거의 그대로 옮긴 문장은 버리지만 뜻이 같은지는 모릅니다.

**거르는 기준**
- 시드 지시와 이미 남긴 지시 모두와 비교해, 하나라도 자카드가 0.8 이상이면 버립니다.
- 원 논문은 ROUGE-L(두 문장에 같은 순서로 나오는 가장 긴 낱말열의 길이로 계산한 유사도, 사이에 다른 낱말이 끼어도 됨)이 0.7 이상이면 버리고, 비교 대상도 이미 모은 지시 전체입니다.

In [ ]:
# 비교 대상: 시드 지시 + 지금까지 남긴 지시
compared_texts = list(seed_instructions)

kept_instructions = []
for instruction in new_instructions:
    # 비교 대상 가운데 하나라도 0.8 이상이면 비슷한 지시로 보고, 더 비교하지 않고 멈춥니다
    is_similar = False
    for other in compared_texts:
        if jaccard(instruction["question"], other) >= 0.8:  # 글자 집합이 80% 이상 겹치면 비슷한 지시
            is_similar = True
            break

    # 비슷하지 않은 지시만 남기고, 다음 지시의 비교 대상에 더합니다
    if not is_similar:
        kept_instructions.append(instruction)
        compared_texts.append(instruction["question"])

print("남은 지시 수:", len(kept_instructions), "/", len(new_instructions))

**결과 읽기**
- **버린 지시가 0일 수 있습니다:** 글자 집합이 80% 이상 겹치는 지시가 드물어서이고, 뜻이 같은 질문까지 없다는 뜻은 아니므로 남은 지시를 몇 개 읽어 봅니다.

## 7. 새 질문에 답 만들기

질문 늘리기로 만든 새 질문에는 답이 없고, 어느 조문에 근거하는지도 정해져 있지 않습니다. 질문마다 관련 있는 조문을 검색으로 찾고, 앞 단계와 같은 생성용 LLM이 그 조문을 보고 근거 조문(`doc_id`)과 답을 씁니다.

### 7-1. 조문 검색기

- **검색하는 이유:** 새 질문의 답은 그 질문의 주제 태그가 아닌 조문에 있을 수 있습니다. 예를 들어 출산전후휴가 뒤 복귀를 묻는 모성보호·육아 질문의 근거인 근로기준법 제74조 제6항은 여성·소년 태그입니다. 408문단 전체에서 질문에 맞는 몇 문단만 찾아 넣으면 주제 태그와 관계없이 근거를 찾고, 408문단을 모두 넣는 것보다 요청의 입력이 짧아 비용이 줄어듭니다.
- **하이브리드 검색:** 키워드 검색(BM25: 질문과 같은 낱말이 많은 문단의 점수가 높음)과 의미 검색(임베딩: 낱말이 달라도 뜻이 가까운 문단을 찾음)의 순위를 RRF로 합쳐 후보를 줍니다. "연차휴가"처럼 조문 용어를 쓴 질문은 키워드 검색이, "쉬는 날"처럼 평소 말로 쓴 질문은 의미 검색이 잘 찾습니다.
- **재정렬(리랭킹)은 선택:** `use_rerank=True`로 켜면 Cross-Encoder가 후보를 질문과 한 쌍씩 읽어 다시 점수를 매겨 순서가 더 정확해집니다. GPU 없이 CPU에서는 질문 하나에 20초 안팎이 걸려 남은 지시 전체에 한 시간 가까이 걸리므로, 이 교안은 수업 시간 관계상 재정렬을 끕니다(`use_rerank=False`).

**검색기 파일 `law_retriever.py`**
- 하이브리드 검색 단원과 리랭킹·컨텍스트 압축 단원에서 만든 LangChain 검색기를 그대로 옮긴 파일로, 단원 폴더에 있습니다. 노트북에서는 `build_retriever`만 불러 씁니다.
- **구성:** `BM25Retriever`(kiwi 형태소 분석으로 명사, 외국어, 숫자만 토큰으로 씀)와 `Chroma` 의미 검색(`text-embedding-3-large`, 768차원)을 `EnsembleRetriever`(가중치 0.5씩, 두 검색 결과의 같은 조문은 `id_key="doc_id"`로 알아봄)로 합칩니다. `use_rerank=False`면 이 `EnsembleRetriever`를 그대로 돌려주고, `invoke(질문)`가 RRF 순서의 Document 리스트를 돌려줍니다. `use_rerank=True`면 `CrossEncoderReranker`(`Qwen/Qwen3-Reranker-0.6B`)를 `ContextualCompressionRetriever`로 이어 재정렬한 상위 `top_n`개를 돌려줍니다.
- **`build_retriever(articles, collection_name, candidate_k, top_n, use_rerank)`:** 키워드 검색과 의미 검색이 각각 `candidate_k`개씩 후보를 찾아 합칩니다. 이 교안은 `candidate_k=5`로 합친 후보(최대 10개) 가운데 RRF 순서의 앞 5개만 씁니다(`[:5]`). 문단이 408개로 많고 답의 조건이 같은 조의 여러 항에 나뉘어 있어 5개를 넣고, 재정렬을 끄면 `top_n`은 쓰지 않습니다.

**비용과 시간**
- **임베딩:** 408문단은 검색기를 만들 때 한 번만 임베딩하고, 검색할 때는 질문 하나만 임베딩합니다. 임베딩 호출은 채팅 모델 호출이 아니라 `get_usage_metadata_callback()`에 잡히지 않아 마지막 비용 테이블에 들어가지 않습니다.
- **검색 시간:** 재정렬을 끄면 리랭커 모델을 내려받지도 부르지도 않으므로, 질문마다 임베딩 요청 1회와 키워드 점수 계산만 합니다. 전체 검색 셀은 50개마다 진행 수를 찍습니다.

In [ ]:
import sys

# 단원 폴더의 검색기 파일(law_retriever.py)을 import 할 수 있게 경로에 더합니다
sys.path.append(str(material_dir))
from law_retriever import build_retriever

# 408문단 전체로 검색기를 만듭니다. 생성 대상에서 뺀 위임 문단과 목적 조항도 다른 조문과 함께 근거가 될 수 있어 넣습니다
# 정리한 본문을 여기서 한 번 임베딩해 Chroma에 넣습니다. 키워드 검색과 의미 검색이 5개씩 찾아 합칩니다(최대 10개)
# 수업 시간 관계상 재정렬은 일부러 끕니다(use_rerank=True로 켜면 순서가 더 정확하지만 GPU 없이는 느립니다)
article_retriever = build_retriever(law_articles, collection_name="labor_law", candidate_k=5, use_rerank=False)
print("검색 대상 문단 수:", len(law_articles))

In [ ]:
# 남은 지시 하나로 검색 결과를 봅니다(질문 임베딩 1회)
sample_question = kept_instructions[0]["question"]
print("질문:", sample_question)
print("질문의 주제 태그:", kept_instructions[0]["source"])
sample_documents = article_retriever.invoke(sample_question)[:5]  # 합친 후보(최대 10개)에서 RRF 순서의 앞 5개만 씁니다

# 검색한 조문을 RRF 순서대로 테이블로 봅니다
retrieved_rows = []
for document in sample_documents:
    retrieved_rows.append({"doc_id": document.metadata["doc_id"], "source": document.metadata["source"], "text": document.page_content})
display(pd.DataFrame(retrieved_rows))

**결과 읽기**
- **순서:** 두 검색의 순위를 RRF로 합친 순서입니다. 질문과 조문을 함께 읽어 다시 매긴 점수가 아니어서 근거 조문이 첫 줄이 아닐 수 있으므로, 5개를 모두 생성용 LLM에 넣습니다.
- **`source` 열:** 질문의 주제 태그와 다른 태그의 조문도 나올 수 있고, 그 조문도 답의 근거가 됩니다.

### 7-2. 답 만들기 체인

- **입력:** 질문마다 검색한 조문 5개를 "doc_id: 본문" 줄로 이어 `{articles}` 자리에 넣습니다. 생성용 LLM은 이 가운데 하나의 `doc_id`를 근거로 고릅니다.
- **doc_id는 글자 그대로(규칙 1, 6):** 답 문장에는 항 번호를 쓰지 않게 하므로, 생성용 LLM이 이 규칙을 doc_id에도 적용해 '근로기준법 제27조'처럼 항을 뺀 id를 돌려줄 수 있습니다. 그러면 검색한 조문의 id와 글자가 달라 답이 있어도 근거 없는 행으로 빠지므로, doc_id는 항 번호까지 그대로 적고 규칙 6은 답 문장에만 적용한다고 나눠 적습니다.
- **핵심에 답할 수 없을 때만 "없음"(규칙 2~4):** 생성용 LLM은 검색한 조문만 보므로 질문의 일부에만 답할 수 있는 경우가 많습니다. 핵심에 답할 수 있으면 조문에 적힌 내용만 조건과 함께 답하게 하고, 거절이나 추측 문장은 넣지 않게 합니다. 거절 문장이 섞인 답으로 파인튜닝하면 모델이 답할 수 있는 질문에도 일부를 거절하기 때문입니다.

In [ ]:
class GroundedAnswer(BaseModel):
    """생성용 LLM이 검색으로 찾은 조문을 보고 쓴 답입니다."""
    doc_id: str = Field(description="답의 근거가 된 조문의 doc_id. [조문] 줄 맨 앞의 doc_id를 항 번호까지 글자 그대로 적음(예: 근로기준법 제60조 제1항). 조문에 답이 없으면 '없음'")
    answer: str = Field(description="조문에 근거한 상담원의 한두 문장 답")


# {articles}에 검색으로 찾은 조문, {question}에 질문을 채웁니다
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "주어진 노동법 조문에 적힌 내용으로만 상담 질문에 답해야 한다. 네가 쓴 답은 노무 상담 챗봇이 배울 예시가 된다."),
    ("human", """아래 [질문]에 [조문]만 근거로 답하라.

근거와 답할 범위:
1. 답의 근거가 된 조문을 하나 고른다. doc_id에는 [조문] 줄 맨 앞의 doc_id를 항 번호까지 글자 그대로 적는다(예: 근로기준법 제60조 제1항).
2. [조문]으로 질문의 핵심에 답할 수 있으면 조문에 적힌 내용만으로 조건을 빠뜨리지 않고 답한다. 질문 가운데 [조문]이 다루지 않는 부분은 답에서 빼고 언급하지 않는다('안내해 드릴 수 없습니다', '제공된 조문만으로는', '정해져 있지 않습니다', '판단할 수 없습니다' 같은 문장을 쓰지 않는다).
3. 자기 상황이 조문의 요건에 해당하는지 묻는 질문이면 조문의 요건을 밝히고 '이런 경우라면 ~에 해당할 수 있습니다'처럼 요건에 맞춰 답한다.
4. [조문]으로 질문의 핵심에 답할 수 없을 때만 doc_id에 "없음"을 적고, 답은 "안내해 드릴 수 없는 질문입니다."로 쓴다.

답 쓰는 법:
5. 답은 상담원이 직접 하는 존댓말로 한두 문장 쓰고, '조문에 따르면'·'조문상'·'이 조문'·'이 조항'·'해당 조항'·'해당 규정' 같은 말은 쓰지 않는다.
6. 답 문장에서 조문을 밝힐 때는 '근로기준법 제60조'처럼 법령 이름과 조 번호만 쓰고, 항·호 번호(제○항, 제○호)는 쓰지 않는다. 이 규칙은 답 문장에만 적용하고, doc_id는 규칙 1대로 적는다.

[조문]
{articles}

[질문]
{question}"""),
])

# 프롬프트 -> 생성용 LLM(GroundedAnswer 모양으로만 답함) 체인
answer_generator = answer_prompt | llm.with_structured_output(GroundedAnswer)

In [ ]:
# 남은 지시마다 조문을 검색하고, 검색한 조문을 "doc_id: 본문" 줄로 이어 체인 입력을 만듭니다
# 질문마다 검색한 doc_id 리스트는 뒤에서 생성용 LLM이 고른 근거를 확인할 때 씁니다
retrieved_doc_ids = []
answer_inputs = []
for index, instruction in enumerate(kept_instructions, start=1):
    # 1) 하이브리드 검색으로 조문 찾기(질문 임베딩 1회)
    documents = article_retriever.invoke(instruction["question"])[:5]  # RRF 순서의 앞 5개만 씁니다
    if index % 50 == 0:  # 질문마다 임베딩 요청을 보내므로 50개마다 진행 수를 찍습니다
        print("검색한 지시 수:", index, "/", len(kept_instructions))

    # 2) 생성용 LLM이 고를 doc_id를 줄 앞에 두고, 줄바꿈으로 잇기
    lines = []
    doc_ids = []
    for document in documents:
        lines.append(f"{document.metadata['doc_id']}: {document.page_content}")
        doc_ids.append(document.metadata["doc_id"])
    retrieved_doc_ids.append(doc_ids)
    answer_inputs.append({"articles": "\n".join(lines), "question": instruction["question"]})

# 보낼 입력 하나를 봅니다: 질문, 검색한 doc_id, {articles}에 들어갈 조문 줄
print("질문:", answer_inputs[0]["question"])
print("검색한 doc_id:", retrieved_doc_ids[0])
print(answer_inputs[0]["articles"])

In [ ]:
# 남은 질문마다 검색한 조문과 함께 한꺼번에 보냅니다. 실패한 요청은 결과 자리에 예외 객체로 돌려받습니다
with get_usage_metadata_callback() as usage:
    answer_results = answer_generator.batch(answer_inputs, config={"max_concurrency": MAX_CONCURRENCY}, return_exceptions=True)
stage_usage["새 질문에 답 만들기"] = usage.usage_metadata
print("보낸 요청 수:", len(answer_inputs))

### 7-3. 근거 없는 질문 나누기

앞에서 늘린 새 질문에는 조문에 없는 내용을 묻는 것도 섞입니다. 생성용 LLM이 고른 `doc_id`가 그 질문에서 검색한 조문 가운데 하나면 `answer_rows`에, "없음"이거나 검색한 조문에 없는 id면 `no_source_rows`에 나눕니다.

답 만들기 프롬프트의 규칙 2로 막아도 근거 있는 답에 거절 문장이 남을 수 있습니다. 근거 doc_id가 검색한 조문이어도 답에 `BANNED_PHRASES`의 표현(거절 문장 포함)이 있으면 `banned_rows`에 따로 모으고 수를 셉니다.

In [ ]:
answer_rows = []
no_source_rows = []
banned_rows = []
failed_count = 0
for instruction, question_doc_ids, result in zip(kept_instructions, retrieved_doc_ids, answer_results):
    # 실패한 요청은 결과 자리에 예외 객체가 있으므로 세기만 합니다
    if isinstance(result, Exception):
        failed_count += 1
    # 근거 doc_id가 그 질문에서 검색한 조문이 아니면("없음" 포함) 근거 없는 행입니다
    elif result.doc_id not in question_doc_ids:
        no_source_rows.append({"source": instruction["source"], "question": instruction["question"], "doc_id": result.doc_id, "answer": result.answer})
    # 근거는 있지만 답에 막은 표현(일부만 답하고 붙인 거절 문장 포함)이 있으면 따로 모읍니다
    elif has_banned_phrase(result.answer):
        banned_rows.append({"source": instruction["source"], "question": instruction["question"], "doc_id": result.doc_id, "answer": result.answer})
    else:
        answer_rows.append({
            "id": f"nq-{len(answer_rows) + 1:04d}",
            "doc_id": result.doc_id,
            "source": instruction["source"],
            "method": "질문 늘리기",
            "question": instruction["question"],
            "answer": result.answer,
        })

print("실패한 요청 수:", failed_count)
print("근거 있는 행 수:", len(answer_rows))
print("근거 없는 행 수:", len(no_source_rows))
print("막은 표현이 있어 뺀 행 수:", len(banned_rows))

# 근거 없는 행(열 이름을 정해 두면 0건이어도 빈 테이블이 만들어집니다)
no_source_table = pd.DataFrame(no_source_rows, columns=["source", "question", "doc_id", "answer"])
display(no_source_table.head(10))

In [ ]:
# 생성용 LLM이 고른 doc_id가 틀렸거나 숫자를 바꿨을 수 있으므로, 근거 있는 행도 숫자 대조를 합니다
answer_table = pd.DataFrame(answer_rows)
answer_table["숫자 대조"] = [numbers_match(row["answer"], doc_texts[row["doc_id"]]) for row in answer_rows]
display(answer_table.head())
display(answer_table["숫자 대조"].value_counts())

**결과 읽기**
- **근거 없는 행:** 숫자 대조를 할 수 없어 학습 데이터에서 뺍니다. "없음"은 검색한 조문 5개로 질문의 핵심에 답할 수 없다는 뜻이라, 조문에 없는 내용을 묻는 질문과 검색이 근거 조문을 놓친 질문이 함께 남습니다.
- **"없음"이 많으면:** 질문을 몇 개 읽어, 근거 조문이 검색 결과에 없었으면 `candidate_k`와 남기는 수(`[:5]`)를 늘리거나 `use_rerank=True`로 재정렬을 켜고, 검색 결과에 있는데도 "없음"이면 답 만들기 프롬프트를 고칩니다.
- **막은 표현이 있어 뺀 행:** 0이 아니면 `banned_rows`의 답을 읽고, 거절 문장이 많으면 답 만들기 프롬프트의 규칙 2를 고칩니다. 조문에 없는 조언은 숫자 대조로 잡히지 않으므로 근거 있는 행도 몇 개 읽습니다.

**이용 약관**
- 상용 LLM 출력을 학습 데이터로 쓰기 전에 그 서비스의 이용 약관을 확인합니다.
- 오픈소스 모델(예: 부록의 Gemma 4)을 생성용 LLM으로 쓰면, 그 모델의 라이선스가 출력을 다른 모델 학습에 쓰는 것을 허용하는지 확인합니다.

### 7-4. 범위 밖 질문의 거절 행

근거 있는 답만 학습한 모델은 이 챗봇이 다루는 다섯 법률로 답할 수 없는 질문에도 단정해서 답하기 쉽습니다. 그래서 모르는 질문에 답하지 않는 예시(거절 행)를 전체의 몇 퍼센트만 섞습니다.

- **앞에서 나눈 근거 없는 행을 쓰지 않는 이유:** 검색이 근거 조문을 놓쳐 "없음"이 된 질문이 섞여 있어, 거절 답을 붙이면 다섯 법률로 답할 수 있는 질문까지 거절하게 됩니다.
- **질문 만들기:** 새 체인 없이 질문 늘리기에서 쓴 `instruction_generator`를 다시 씁니다. `topic`에 범위 밖 주제 `OUT_OF_SCOPE_TOPIC`을 넣고 같은 시드로 질문 70개를 요청합니다(호출 1회).
- **요청 개수 70(`OUT_OF_SCOPE_COUNT`):** 교안 02는 근거 있는 행만 증강하고 기존 FAQ를 더하므로 범위 밖 행의 비율이 낮아집니다. 증강과 필터 뒤에도 범위 밖 행이 전체의 3~5%가 되도록 70개를 요청합니다.
- **앞에서부터 70개만:** 스키마가 개수를 강제하지 않아 70개보다 많이 올 수 있으므로, 받은 질문 리스트를 `[:OUT_OF_SCOPE_COUNT]`로 잘라 씁니다. 적게 오면 받은 만큼만 씁니다.
- **`question_type` 키 없음:** 질문 유형은 조문으로 Q&A 생성 체인(`qa_generator`)만 붙이므로, 질문 늘리기 행과 거절 행에는 이 키가 없습니다.

In [ ]:
# 이 챗봇이 다루는 다섯 법률로 답할 수 없는 주제와, 그 질문에 붙일 거절 답
OUT_OF_SCOPE_TOPIC = "이 챗봇이 다루는 다섯 법률(근로기준법, 최저임금법, 퇴직급여법, 남녀고용평등법, 기간제법)로는 답할 수 없는 질문(연말정산 같은 세금, 실업급여와 고용보험, 4대보험 가입, 이직 준비(면접, 경력기술서, 연봉 협상), 개인 투자)"
REFUSAL_ANSWER = "이 상담 챗봇이 다루는 노동법 조문으로는 답할 수 없는 질문입니다. 이 내용은 담당 기관이나 전문가에게 문의해 주세요."

# 요청할 범위 밖 질문 수: 교안 02의 증강과 필터 뒤에도 범위 밖 행이 3~5%가 되게 하는 개수입니다
OUT_OF_SCOPE_COUNT = 70

# 질문 늘리기에서 쓴 instruction_generator로 범위 밖 질문을 요청합니다(호출 1회)
print("{topic} 자리에 넣을 범위 밖 주제:", OUT_OF_SCOPE_TOPIC)
with get_usage_metadata_callback() as usage:
    out_of_scope_result = instruction_generator.invoke({"topic": OUT_OF_SCOPE_TOPIC, "seeds": seed_lines, "n": OUT_OF_SCOPE_COUNT})
stage_usage["범위 밖 질문 만들기"] = usage.usage_metadata

# 질문마다 거절 답을 붙여 행으로 모읍니다
# 스키마가 개수를 강제하지 않아 더 많이 올 수 있으므로, 앞에서부터 OUT_OF_SCOPE_COUNT개만 씁니다
refusal_rows = []
for question in out_of_scope_result.items[:OUT_OF_SCOPE_COUNT]:
    refusal_rows.append({
        "id": f"out-{len(refusal_rows) + 1:04d}",
        "doc_id": "없음",
        "source": "범위 밖",
        "method": "범위 밖",
        "question": question,
        "answer": REFUSAL_ANSWER,
    })

# 사람이 질문을 모두 읽을 수 있게 거절 행 전체를 테이블로 봅니다(doc_id, source, method, answer는 모든 행이 같으므로 id와 question 열만 봅니다)
# 테이블이 60행을 넘으면 앞뒤 몇 행만 보이므로, 보여 줄 최대 행 수를 200으로 늘립니다
pd.set_option("display.max_rows", 200)
refusal_table = pd.DataFrame(refusal_rows)
print("받은 질문 수:", len(out_of_scope_result.items), "| 거절 행 수:", len(refusal_rows))
display(refusal_table[["id", "question"]])

**결과 읽기**
- **사람이 읽을 것:** 다섯 법률로 답할 수 있는 질문(예: 퇴직금, 연차)이 섞였으면 그 행을 `refusal_rows`에서 뺍니다. 그런 질문에 거절 답을 학습하면 답해야 할 질문까지 거절합니다.
- **비율:** 거절 행의 비율은 모든 행을 저장한 뒤 교안 02에서 확인합니다.

## 8. 저장

생성을 모두 마쳤으므로 단계별 API 비용을 확인하고, 만든 행을 파일로 저장합니다.

### 8-1. API 비용 확인

API를 부른 단계마다 `get_usage_metadata_callback()`으로 받은 토큰 사용량을 `stage_usage`에 모았습니다. 단계마다 `usage_cost`로 실제 비용을 구하고, 조문으로 Q&A 생성은 5개 문단 시험으로 어림한 예상 비용과 비교합니다. 연차 조문 하나로 만든 호출 1회, 검색기의 임베딩 호출, 함께 따라하기의 호출은 이 테이블에 넣지 않았습니다.

In [ ]:
# 단계마다 usage_cost로 실제 비용을 계산합니다
cost_rows = []
for stage, usage_metadata in stage_usage.items():
    cost_rows.append({"단계": stage, "실제 비용(달러)": round(usage_cost(usage_metadata), 4)})
cost_table = pd.DataFrame(cost_rows)
display(cost_table)
print("실제 비용 합계(달러):", round(cost_table["실제 비용(달러)"].sum(), 4))

# 조문으로 Q&A 생성은 5개 문단 시험으로 어림한 예상 비용과 비교합니다
print("조문으로 Q&A 생성 예상(달러):", round(estimated_cost, 4), "| 실제(달러):", round(usage_cost(stage_usage["조문으로 Q&A 생성"]), 4))

**결과 읽기**
- **예상과 실제:** 출력의 추론 토큰은 요청마다 길이가 달라 비용을 흔들고, 캐시 입력은 싸게 계산되어 실제 비용을 낮춥니다.
- **어림하지 않은 단계:** 질문 늘리기, 새 질문에 답 만들기, 범위 밖 질문 만들기는 실제 비용만 봅니다. 새 질문에 답 만들기는 요청마다 검색한 조문 5개가 들어가, 문단 하나를 넣는 조문으로 Q&A 생성보다 요청당 입력 토큰 수가 많습니다.
- **실패한 요청:** 끝내 실패한 요청은 사용량이 기록되지 않아 실제 비용에 들어가지 않습니다.

### 8-2. JSONL 저장

교안 02에서 이어서 다듬으려면 만든 행을 파일로 남겨야 합니다. 문서 생성 행과 질문 늘리기 행을 합쳐 숫자 대조를 통과하고 금지 표현이 없는 행만 남기고, 거절 행을 더해 JSONL로 저장합니다.

In [ ]:
# 숫자 대조를 통과하고 막은 표현이 없는 행만 남깁니다
dataset_rows = []
for row in qa_rows + answer_rows:
    if numbers_match(row["answer"], doc_texts[row["doc_id"]]) and not has_banned_phrase(row["answer"]):
        dataset_rows.append(row)

# 거절 행은 근거 조문이 없어 숫자 대조 대상이 아니므로 검수 뒤에 더합니다
dataset_rows = dataset_rows + refusal_rows
print("저장할 행 수:", len(dataset_rows))

In [ ]:
# 한 줄에 행 하나씩 JSON으로 씁니다(키: id, doc_id, source, method, question, answer, 문서 생성 행은 question_type 도)
dataset_path = output_dir / "labor_qa.jsonl"
with open(dataset_path, "w", encoding="utf-8") as file:
    for row in dataset_rows:
        file.write(f"{json.dumps(row, ensure_ascii=False)}\n")
print("저장한 파일:", dataset_path)

# 방법별 저장 행 수
dataset_table = pd.DataFrame(dataset_rows)
display(dataset_table["method"].value_counts())

교안 02는 이 파일(`output/labor_qa.jsonl`)을 읽어 증강, 품질 필터, 오염 검사, 구성 균형, chat 포맷 변환, 학습/검증 분할을 이어 갑니다.

## 9. 🖐️ 함께 따라하기: 전월세 상담 Q&A 만들기

본실습의 `clean_text`, `qa_generator`, `numbers_match`를 다른 법령에 적용해, 원문 정리 -> Q&A 행 -> 숫자 대조가 그대로 도는지 확인합니다. 이번에는 생성이 중간에 멈춰도 멈춘 곳부터 이어서 처리하는 구조를 직접 작성하고, 마지막에 본실습의 검색기 파일(`law_retriever.py`)로 전월세 조문 검색기를 만들어 세입자 질문의 근거 조문을 찾습니다.

**본실습과 다른 점**
- **데이터:** 주택임대차보호법 45문단(`data/따라하기/lease_articles.json`)입니다. 본실습과 같은 국가법령정보센터 원문이고 키도 같습니다. `source`는 보증금 반환·임차권등기, 계약 기간·갱신, 대항력·우선변제, 적용 범위·기타, 차임·보증금 증감 다섯 가지입니다.
- **규모:** 45문단이고 같은 체인을 본실습에서 이미 시험했으므로 5개 문단 시험 없이 보냅니다.
- **이어서 생성:** 생성 결과를 묶음마다 진행 파일에 이어 쓰고, 다시 실행하면 끝난 문단을 건너뜁니다. 실패한 요청이나 커널 중단 뒤에도 남은 문단만 이어서 보냅니다. 수백 개 요청을 보내는 실무 생성은 중간에 멈추는 일이 흔하고, 처음부터 다시 보내면 멈춘 곳에 따라 비용과 시간이 최대 두 배까지 듭니다.
- **id:** 진행 파일에는 id 없이 쓰고, 생성이 끝난 뒤 진행 파일 전체를 읽을 때 `lqa-001`부터 순번으로 붙입니다. 여러 번 나눠 실행해 이어 쓴 행도 번호가 겹치지 않습니다.
- **검색기:** 45문단이라 후보를 3개씩 찾아(`candidate_k=3`, 합치면 최대 6개) RRF 순서의 앞 3개만 씁니다(`[:3]`). 재정렬은 본실습과 같이 끄고(`use_rerank=False`), 생성용 LLM은 부르지 않고 답도 만들지 않습니다.

**실습 진행 방법:** 원문 정리, 생성 대상 고르기, 숫자 대조 셀은 본실습 함수를 그대로 부르는 완성 코드이므로 실행만 합니다. 직접 작성하는 곳은 "진행 파일에 이어 쓰며 생성하기"의 세 셀(끝난 문단 모으기, 보낼 문단 고르기, 묶음 생성과 이어 쓰기)과 "검색기로 근거 조문 찾기"의 한 셀(검색기 만들기, 검색 호출)이고, `...`를 바로 위 주석에 맞춰 채웁니다. 완성 코드는 `정답/교안_01_정답.ipynb`의 같은 절에 있습니다.

### 9-1. 조문 읽기와 정리

In [ ]:
# 주택임대차보호법 45문단을 읽습니다
lease_articles = json.loads((data_dir / "따라하기" / "lease_articles.json").read_text(encoding="utf-8"))

# 문단마다 본문을 본실습의 clean_text로 정리합니다
for article in lease_articles:
    article["text"] = clean_text(article["text"])

# 개정 태그가 남은 문단 수(0이면 정리된 것입니다)
tag_count = 0
for article in lease_articles:
    if "<" in article["text"]:  # 개정 태그는 <개정 ...> 모양이라 꺾쇠로 찾습니다
        tag_count += 1
print("문단 수:", len(lease_articles))
print("개정 태그가 남은 문단 수:", tag_count)

lease_table = pd.DataFrame(lease_articles)
display(lease_table.head())

본실습에서 만든 `is_qa_target`으로 위임 문장 하나뿐인 문단과 목적 조항을 빼고 생성 대상 문단을 고릅니다.

In [ ]:
# 본실습의 is_qa_target으로 생성 대상 문단만 lease_qa_articles에 모읍니다
lease_qa_articles = []
for article in lease_articles:
    if is_qa_target(article["text"]):
        lease_qa_articles.append(article)
print("생성 대상 문단 수:", len(lease_qa_articles), "/", len(lease_articles))

### 9-2. 진행 파일에 이어 쓰며 생성하기

`.batch`는 모든 요청이 끝나야 결과를 돌려주므로, 본실습처럼 생성 대상 전체를 한 번에 보내면 중간에 커널이 멈출 때 끝난 요청의 결과도 받지 못합니다. 여기서는 문단을 묶음으로 나눠 보내고, 묶음이 끝날 때마다 성공한 행을 진행 파일에 이어 씁니다.

- **진행 파일(`PROGRESS_PATH`):** 생성한 행을 한 줄에 하나씩 적는 JSONL 파일(`output/lease_qa_progress.jsonl`)입니다. 이 파일에 `doc_id`가 있는 문단은 끝난 문단입니다.
- **처음부터 다시 만들 때:** `output/lease_qa_progress.jsonl` 파일을 지우고 실행합니다.

먼저 진행 파일에서 끝난 문단을 모읍니다.

- **건너뛰기:** 진행 파일이 있으면(`PROGRESS_PATH.exists()`) 줄마다 `json.loads`로 읽어 `doc_id`를 집합(set) `done_doc_ids`에 모으고, 이 집합에 없는 문단만 보냅니다. 실패한 문단은 파일에 쓰지 않으므로 다음 실행에서 다시 보냅니다.

In [ ]:
# 🖐️ 함께 따라하기: 진행 파일에서 끝난 문단의 doc_id를 모으세요

# 생성한 행을 한 줄씩 이어 쓰는 진행 파일과, .batch 한 번에 보낼 문단 수
PROGRESS_PATH = output_dir / "lease_qa_progress.jsonl"
CHUNK_SIZE = 15

# 진행 파일이 있으면 줄마다 읽어, 끝난 문단의 doc_id를 done_doc_ids에 모으기
done_doc_ids = set()
if PROGRESS_PATH.exists():  # 첫 실행에는 진행 파일이 없어 건너뜁니다
    with open(PROGRESS_PATH, encoding="utf-8") as file:
        for line in file:
            # 줄 하나(line)를 json.loads로 행 딕셔너리로 바꾸기
            row = ...
            # 그 행의 doc_id를 done_doc_ids에 더하기
            done_doc_ids.add(...)
print("끝난 문단 수:", len(done_doc_ids))

생성 대상 문단 가운데 `done_doc_ids`에 없는 문단만 이번에 보낼 문단으로 남깁니다.

In [ ]:
# 🖐️ 함께 따라하기: 아직 끝나지 않은 문단만 pending_articles에 모으세요

pending_articles = []
for article in lease_qa_articles:
    # 문단의 doc_id가 done_doc_ids에 없는지 검사하는 조건 쓰기
    if ...:
        pending_articles.append(article)
print("보낼 문단 수:", len(pending_articles))

남은 문단을 묶음으로 나눠 보내고, 묶음이 끝날 때마다 진행 파일 끝에 이어 씁니다.

- **`CHUNK_SIZE`(15):** `.batch` 한 번에 보내는 문단 수입니다. `range(0, 문단 수, CHUNK_SIZE)`가 묶음의 시작 번호 0, 15, 30을 주고, 리스트를 `[시작:시작 + CHUNK_SIZE]`로 잘라 묶음을 만듭니다.
- **`open(경로, "a")`:** 파일 끝에 이어 쓰는 모드이고, 파일이 없으면 새로 만듭니다. 본실습에서 저장할 때 쓴 `"w"`는 파일을 비우고 새로 쓰므로 이어 쓰기에 맞지 않습니다.
- **묶음마다 열고 닫기:** 묶음마다 `with`로 파일을 열고 닫으므로, 커널이 멈춰도 이미 끝난 묶음의 행은 파일에 남습니다. 커널이 멈춘 묶음부터 남은 묶음까지만 다음 실행에서 보냅니다.

In [ ]:
# 🖐️ 함께 따라하기: 남은 문단을 CHUNK_SIZE개씩 보내고, 묶음이 끝날 때마다 진행 파일 끝에 이어 쓰세요

failed_doc_ids = []
for chunk_start in range(0, len(pending_articles), CHUNK_SIZE):  # 묶음의 시작 번호 0, 15, 30
    chunk_articles = pending_articles[chunk_start:chunk_start + CHUNK_SIZE]  # 시작 번호부터 15개(끝 묶음은 남은 만큼)

    # 본실습과 같은 모양의 입력 딕셔너리(n=3)
    chunk_inputs = [{"n": 3, "paragraph": article["text"]} for article in chunk_articles]  # n=3: 문단마다 Q&A 최대 3개

    # qa_generator.batch로 chunk_inputs 보내기: 동시 요청 상한은 MAX_CONCURRENCY, 실패해도 나머지 결과를 받는 옵션까지 넣기
    chunk_results = ...

    # 이번 묶음에서 실패한 문단 수
    chunk_failed_count = 0

    # 진행 파일을 파일 끝에 이어 쓰는 모드로 열기: 두 번째 인자에 이어 쓰기 모드 문자열 쓰기
    with open(PROGRESS_PATH, ..., encoding="utf-8") as file:
        for article, result in zip(chunk_articles, chunk_results):
            if isinstance(result, Exception):
                # 실패한 문단의 doc_id를 failed_doc_ids에 더하기
                failed_doc_ids.append(...)
                chunk_failed_count += 1
            else:
                for item in result.items:
                    # 행 딕셔너리(id는 진행 파일 전체를 다시 읽을 때 붙입니다)
                    row = {
                        "doc_id": article["doc_id"],
                        "source": article["source"],
                        "method": "문서 생성",
                        "question": item.question,
                        "answer": item.answer,
                        "question_type": item.question_type,
                    }
                    file.write(f"{json.dumps(row, ensure_ascii=False)}\n")  # 한 줄에 행 하나, 한글은 글자 그대로
    # 번호는 이번 실행에서 보낼 문단(pending_articles) 안의 순번이고, 보낸 문단의 첫 doc_id와 끝 doc_id를 함께 찍습니다
    print("이번 실행 묶음:", chunk_start + 1, "~", chunk_start + len(chunk_articles), "| 실패:", chunk_failed_count, "| 문단:", chunk_articles[0]["doc_id"], "~", chunk_articles[-1]["doc_id"])

# 이번 실행에서 실패한 문단 수(다시 실행하면 이 문단만 보냅니다)
print("이번 실패 문단 수:", len(failed_doc_ids))

### 9-3. 모은 결과로 숫자 대조

진행 파일에는 여러 번 나눠 실행한 결과가 이어 쓰여 있으므로, 파일 전체를 읽어 Q&A 행을 모읍니다. id는 이때 읽은 순서대로 `lqa-001`부터 붙이고, 숫자 대조는 본실습에서 전체 Q&A를 검수한 방법과 같습니다.

In [ ]:
# 진행 파일 전체를 읽어 Q&A 행에 id를 붙이고, 숫자 대조 테이블을 만듭니다

# 1) 진행 파일을 줄마다 JSON으로 읽어 행 딕셔너리로 바꾸고, 읽은 순서대로 id를 붙여 lease_qa_rows에 모읍니다
lease_qa_rows = []
with open(PROGRESS_PATH, encoding="utf-8") as file:
    for line in file:
        row = json.loads(line)
        # id는 lqa-001, lqa-002처럼 세 자리 순번입니다
        row["id"] = f"lqa-{len(lease_qa_rows) + 1:03d}"
        lease_qa_rows.append(row)

# 2) 행 리스트를 테이블로 만들고, 행마다 근거 조문과 숫자를 대조한 결과를 "숫자 대조" 열로 붙입니다
lease_texts = {article["doc_id"]: article["text"] for article in lease_articles}  # doc_id -> 정리한 본문
lease_check = pd.DataFrame(lease_qa_rows)
lease_check["숫자 대조"] = [numbers_match(row["answer"], lease_texts[row["doc_id"]]) for row in lease_qa_rows]

# 걸린 행의 답을 조문과 바로 비교하도록 근거 조문을 열로 붙입니다
lease_check["근거 조문"] = [lease_texts[row["doc_id"]] for row in lease_qa_rows]

print("Q&A 행 수:", len(lease_qa_rows))
display(lease_check.head())

# 숫자 대조에 걸린 행: 답의 숫자를 근거 조문 열과 비교합니다
display(lease_check[lease_check["숫자 대조"] == False])

### 9-4. 검색기로 근거 조문 찾기

본실습의 `build_retriever`로 전월세 조문 45문단의 검색기를 만들고, 세입자 질문 두 개의 근거 조문을 찾습니다. 임베딩은 45문단을 검색기를 만들 때 한 번, 질문은 검색할 때마다 하나씩 합니다.

- **`build_retriever(articles, collection_name, candidate_k, use_rerank)`:** 본실습과 같은 함수입니다. `collection_name`은 본실습 검색기(`"labor_law"`)와 다른 이름(`"lease_law"`)으로 줘야 본실습의 Chroma 컬렉션을 비우지 않습니다.
- **`검색기.invoke(질문)`:** 재정렬을 끈 검색기는 두 검색의 후보를 RRF 순서로 합친 Document 리스트(최대 `candidate_k`의 2배)를 돌려주고, 앞 3개만 쓰려면 `[:3]`으로 자릅니다. 조문 번호는 `document.metadata["doc_id"]`, 본문은 `document.page_content`에 있습니다.

In [ ]:
# 🖐️ 함께 따라하기: 전월세 조문 검색기를 만들고, 세입자 질문마다 근거 조문을 찾으세요

# 세입자가 물을 법한 질문 두 개
lease_questions = [
    "계약이 끝났는데 집주인이 보증금을 안 돌려줘요. 이사를 가야 하는데 보증금을 지킬 방법이 있나요?",
    "전세 계약서에 기간을 1년으로 적었는데, 1년이 지나면 집주인이 나가라고 할 때 나가야 하나요?",
]

# build_retriever로 lease_articles 전체의 검색기 만들기: collection_name은 "lease_law", 후보는 3개씩(candidate_k=3)
# 수업 시간 관계상 재정렬은 일부러 끕니다(use_rerank=True로 켜면 순서가 더 정확하지만 GPU 없이는 느립니다)
lease_retriever = ...

for question in lease_questions:
    # lease_retriever의 invoke로 질문 하나의 근거 조문을 찾고, RRF 순서의 앞 3개만 남기기([:3])
    lease_documents = ...

    # 찾은 조문의 doc_id와 본문을 RRF 순서대로 찍습니다
    print("질문:", question)
    for document in lease_documents:
        print("-", document.metadata["doc_id"], "|", document.page_content)
    print()

**확인 기준**
- **정리와 고르기:** 45문단에 개정 태그가 남은 문단이 0이고, 생성 대상 문단 수가 45 / 45입니다(위임 문단과 목적 조항이 없음).
- **첫 실행:** 진행 파일이 없을 때 세 셀을 차례로 실행하면 끝난 문단 수 0, 보낼 문단 수 45이고, `이번 실행 묶음:`이 1 ~ 15, 16 ~ 30, 31 ~ 45로 세 번 찍힙니다.
- **다시 실행:** 세 셀을 다시 실행하면 보낼 문단 수는 직전 실행에서 실패했거나 Q&A를 하나도 받지 못한 문단 수입니다(없었으면 0). 숫자 대조 셀을 다시 실행해도 Q&A 행 수는 줄지 않습니다.
- **중단 뒤 실행:** 생성 중 커널을 멈췄다가 다시 실행하면 `이번 실행 묶음:`이 찍힌 묶음의 문단은 보내지 않습니다. 번호는 이번에 보낼 문단 안의 순번이라 1부터 셉니다.
- **`lease_check`:** 문단마다 1~3개를 요청해 대개 45~135행이고, 행마다 `doc_id`, `source`, `method`, `question`, `answer`, `question_type`, `id`가 채워져 있습니다.
- **숫자 대조 `False`:** `근거 조문` 열과 비교해 틀린 숫자인지, 단위만 바꾼 맞는 답인지 판정합니다.
- **검색기:** 질문마다 `-` 줄이 3개씩 찍히고, 줄 앞의 `doc_id`가 모두 `주택임대차법`으로 시작합니다(본실습의 노동법 조문이 섞이지 않음).
- **근거 조문:** 찍힌 본문을 읽어, 첫 질문에는 보증금 반환이나 임차권등기명령 조문이, 둘째 질문에는 임대차기간 조문이 들어 있는지 확인합니다. 없으면 `candidate_k`와 남기는 수(`[:3]`)를 늘려 다시 찾습니다.

## 10. 핵심 정리

| 단계 | 하는 일 | 핵심 코드·기준 |
|---|---|---|
| 원문 정리 | 개정 태그 제거, NFC 정규화, 가운뎃점과 공백 정리 | `clean_text`, 408문단 가운데 237문단이 바뀜 |
| 생성 대상 고르기 | 위임만 하는 한 문장과 목적 조항을 뺌 | `is_qa_target`, 408문단 가운데 382문단 |
| 조문으로 Q&A 생성 | 조문 하나에서 Q&A 1~3개, 5개 문단 시험 뒤 전체를 한꺼번에 보냄 | `qa_generator`(`qa_prompt`와 `QASet` 구조화 출력), `.batch`의 `max_concurrency`, `return_exceptions=True` |
| 근거 검수 | 답의 숫자를 근거 조문과 대조하고 막은 표현을 거름 | `numbers_match`, `has_banned_phrase` |
| 질문 늘리기 | 시드 지시 3개로 주제마다 중앙값까지 모자란 만큼 10~30개, 거의 같은 질문은 버림 | `instruction_generator`, `MIN_NEW_PER_TOPIC`, `MAX_NEW_PER_TOPIC`, `jaccard` 0.8 |
| 새 질문에 답 만들기 | 하이브리드 검색(재정렬은 끔)으로 찾은 조문의 앞 5개를 보고 근거 doc_id와 답을 씀, 근거가 검색한 조문에 없거나 "없음"이면 뺌 | `build_retriever(..., use_rerank=False)`(`law_retriever.py`), `article_retriever`, `retrieved_doc_ids`, `answer_generator`, `answer_rows`, `no_source_rows` |
| 범위 밖 질문 만들기 | 다섯 법률로 답할 수 없는 질문에 고정 거절 답 | `OUT_OF_SCOPE_COUNT`(70), `REFUSAL_ANSWER` |
| 비용 확인 | 시험 비용으로 전체를 어림하고, 단계별 실제 비용을 계산 | `usage_cost`, `estimated_cost`, `stage_usage` |
| 저장 | 검수를 통과한 행과 거절 행을 JSONL로 | `output/labor_qa.jsonl` |
| 이어서 생성(함께 따라하기) | 묶음마다 진행 파일에 이어 쓰고 끝난 문단은 건너뜀 | `open(PROGRESS_PATH, "a")`, `done_doc_ids` |
| 근거 조문 검색(함께 따라하기) | 같은 검색기 파일로 전월세 조문 검색기를 만들어 질문의 근거 조문을 찾음 | `build_retriever(lease_articles, ...)`, `lease_retriever.invoke` |

**꼭 기억할 것**
- 숫자 대조는 숫자를 바꾼 답만 잡으므로, 숫자 없는 할루시네이션은 사람이 표본을 읽어 확인합니다.
- 응답 방식(말투, 답의 길이, 거절)은 파인튜닝으로, 바뀔 수 있는 법령 값은 RAG로 넣습니다.
- 실패한 요청은 `return_exceptions=True`로 예외 객체로 받아 나머지 결과를 지키고, 실패한 것만 다시 보냅니다.
- 범위 밖 질문에 거절 답을 학습시켜야 모델이 모르는 질문에 지어내 답하지 않습니다.